# Modelado territorial de la oferta de educación superior en Paraguay

### Cuaderno de tesis — pipeline reproducible de análisis territorial

Este cuaderno implementa el pipeline completo de la investigación *"Modelado
Territorial de la Oferta de Educación Superior en Paraguay mediante Ciencia
de Datos"*. A partir de cuatro fuentes oficiales —el Registro Nacional de
Carreras del CONES, el Censo Nacional de Población y Viviendas 2022 del INE,
un GeoJSON departamental y las Estimaciones y Proyecciones Departamentales
del INE (Revisión 2025)—, el notebook construye el Índice de Cobertura
Relativa (IC), evalúa la pertinencia territorial de la oferta académica
respecto a los ejes productivos, segmenta los departamentos mediante
aprendizaje no supervisado, calcula la evolución oficial de la demanda
potencial y las brechas de oferta hacia 2032, y sintetiza todo en el Índice
de Prioridad Territorial Educativa (IPTE).

El análisis se organiza en 9 pasos secuenciales (ver Sección 16), y aplica
tres criterios metodológicos explícitos a lo largo de todo el pipeline:

- **Asunción se trata como valor atípico**: se reporta de forma descriptiva,
  pero se excluye antes de estandarizar variables, de la segmentación por
  clusters, de los umbrales por cuartiles y del IPTE.
- **El número de grupos (K) se selecciona dinámicamente** mediante el método
  del codo y el coeficiente de silueta sobre los 17 departamentos restantes.
  Si la silueta favoreciera K=2, el pipeline prescindiría de K-means y
  usaría la clasificación por cuartiles del IC como criterio sustantivo.
- **El componente prospectivo usa exclusivamente proyecciones oficiales**
  del INE (2022–2032, grupo etario 15–29 años); no se ajusta ninguna
  regresión demográfica propia.

Todas las salidas (tablas, figuras y mapas) se exportan a la carpeta
`resultados_tesis/` y corresponden a las tablas y figuras citadas en los
Capítulos 3 y 4 de la tesis.

## 1. Instalación de dependencias

Google Colab trae preinstaladas la mayoría de las librerías que necesita el
pipeline (`pandas`, `numpy`, `matplotlib`, `scikit-learn`, `scipy`). Las
librerías **geoespaciales** (`geopandas`, `folium`) y las de **estadística
espacial** (`libpysal`, `esda`, empleadas para el cálculo del Índice de
Moran) no forman parte del entorno base de Colab, por lo que esta celda las
instala mediante `pip`.

Esta celda solo necesita ejecutarse una vez por sesión de Colab. Si alguna
de las librerías ya estuviera instalada, `pip` lo detecta automáticamente y
no vuelve a descargarla.


In [ ]:
# Instalación de librerías geoespaciales y de estadística espacial
!pip install -q geopandas folium libpysal esda


## 2. Importación de librerías

Se importan herramientas para manejo de datos, gráficos, geoprocesamiento y
segmentación: pandas, NumPy, Matplotlib, SciPy, K-means, silueta y
estandarización. Las dependencias geoespaciales son opcionales; si no están
disponibles, el análisis tabular continúa y solo se omiten los mapas y Moran.
No se importa ningún modelo de proyección demográfica.


In [ ]:
from __future__ import annotations

import json
import math
import os
import re
import unicodedata
import warnings
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Set, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D
from scipy.cluster.hierarchy import dendrogram, fcluster, linkage
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

try:
    import geopandas as gpd
except ImportError:
    gpd = None

try:
    import folium
except ImportError:
    folium = None

try:
    from libpysal.weights import Queen
    from esda.moran import Moran
except ImportError:
    Queen = None
    Moran = None

print("Librerías importadas correctamente.")
print(f"geopandas disponible: {gpd is not None}")
print(f"folium disponible: {folium is not None}")
print(f"libpysal/esda disponibles (Índice de Moran): {Queen is not None and Moran is not None}")


## 3. Configuración general

Se definen las rutas de los cuatro archivos de entrada, las carpetas de
resultados, el año base 2022, el horizonte oficial 2022–2032 y el rango K=2,…,8.
`RANDOM_STATE` asegura reproducibilidad y
`DEPARTAMENTO_EXCLUIDO_CLUSTERING="ASUNCION"` fija la exclusión metodológica
antes del escalamiento. También se declaran los 18 territorios, los datos
censales usados para validación, los ejes productivos y la población 15–29
observada en el Censo 2022.

Antes de ejecutar, las rutas deben coincidir con los nombres exactos de los
archivos subidos a Colab.


In [ ]:
RUTA_CONES = "/content/carreras.txt"
RUTA_GEOJSON   = "/content/geojson.txt"
RUTA_INE_CUADRO_1  = "/content/Cuadro_1.xlsx"
RUTA_INE_PROYECCIONES = "/content/Estimaciones y Proyecciones Departamentales. Revisión 2025.xlsx"

CARPETA_SALIDA = Path("resultados_tesis")
CARPETA_FIGURAS = CARPETA_SALIDA / "figuras"
CARPETA_MAPAS = CARPETA_SALIDA / "mapas"

ANIO_BASE = 2022
ANIOS_ANALISIS = list(range(2022, 2033))
RANDOM_STATE = 42
K_MIN = 2
K_MAX = 8
DEPARTAMENTO_EXCLUIDO_CLUSTERING = "ASUNCION"
np.random.seed(RANDOM_STATE)

DEPARTAMENTOS = [
    "ASUNCION", "CONCEPCION", "SAN PEDRO", "CORDILLERA", "GUAIRA",
    "CAAGUAZU", "CAAZAPA", "ITAPUA", "MISIONES", "PARAGUARI",
    "ALTO PARANA", "CENTRAL", "NEEMBUCU", "AMAMBAY", "CANINDEYU",
    "PRESIDENTE HAYES", "BOQUERON", "ALTO PARAGUAY",
]

NOMBRES_DISPLAY = {
    "ASUNCION": "Asuncion",
    "CONCEPCION": "Concepcion",
    "SAN PEDRO": "San Pedro",
    "CORDILLERA": "Cordillera",
    "GUAIRA": "Guaira",
    "CAAGUAZU": "Caaguazu",
    "CAAZAPA": "Caazapa",
    "ITAPUA": "Itapua",
    "MISIONES": "Misiones",
    "PARAGUARI": "Paraguari",
    "ALTO PARANA": "Alto Parana",
    "CENTRAL": "Central",
    "NEEMBUCU": "Neembucu",
    "AMAMBAY": "Amambay",
    "CANINDEYU": "Canindeyu",
    "PRESIDENTE HAYES": "Presidente Hayes",
    "BOQUERON": "Boqueron",
    "ALTO PARAGUAY": "Alto Paraguay",
}

# Poblacion total por departamento de los censos 1982, 1992, 2002, 2012 y 2022.
# Fuente: Instituto Nacional de Estadistica (INE) - Censos Nacionales de
# Poblacion y Viviendas 1982, 1992, 2002, 2012 y 2022. Valores oficiales confirmados.
CENSOS_DEPARTAMENTALES = {
    "ASUNCION":      {1982: 454881, 1992: 500938, 2002: 512112, 2012: 529433, 2022: 462241},
    "CONCEPCION":    {1982: 133977, 1992: 167289, 2002: 179450, 2012: 226585, 2022: 206299},
    "SAN PEDRO":     {1982: 191002, 1992: 280336, 2002: 318698, 2012: 394169, 2022: 355244},
    "CORDILLERA":    {1982: 140011, 1992: 178701, 2002: 233854, 2012: 279860, 2022: 268133},
    "GUAIRA":        {1982: 143550, 1992: 161991, 2002: 178650, 2012: 209900, 2022: 179561},
    "CAAGUAZU":      {1982: 299437, 1992: 386412, 2002: 435357, 2012: 518218, 2022: 431453},
    "CAAZAPA":       {1982: 109452, 1992: 129352, 2002: 139517, 2012: 172345, 2022: 139484},
    "ITAPUA":        {1982: 262680, 1992: 377536, 2002: 453692, 2012: 554653, 2022: 449682},
    "MISIONES":      {1982: 77475, 1992: 89018, 2002: 101783, 2012: 116672, 2022: 111133},
    "PARAGUARI":     {1982: 204399, 1992: 208527, 2002: 221932, 2012: 248461, 2022: 200522},
    "ALTO PARANA":   {1982: 199644, 1992: 406584, 2002: 558672, 2012: 737092, 2022: 764026},
    "CENTRAL":       {1982: 497388, 1992: 866856, 2002: 1362893, 2012: 1855241, 2022: 1884288},
    "NEEMBUCU":      {1982: 70338, 1992: 69770, 2002: 76348, 2012: 86180, 2022: 76697},
    "AMAMBAY":       {1982: 68395, 1992: 99860, 2002: 114917, 2012: 151395, 2022: 179419},
    "CANINDEYU":     {1982: 66409, 1992: 103185, 2002: 140137, 2012: 198899, 2022: 191186},
    "PRESIDENTE HAYES": {1982: 33021, 1992: 64417, 2002: 82493, 2012: 109818, 2022: 123357},
    "BOQUERON":      {1982: 14190, 1992: 29060, 2002: 41106, 2012: 56440, 2022: 71085},
    "ALTO PARAGUAY": {1982: 9021, 1992: 12156, 2002: 11587, 2012: 15682, 2022: 17203},
}

# Ejes productivos requeridos, segun la matriz heuristica documentada en la tesis.
EJES_PRODUCTIVOS = {
    "ALTO PARAGUAY": {"AMBIENTE", "GANADERIA", "LOGISTICA"},
    "ALTO PARANA": {"COMERCIO", "INDUSTRIA", "LOGISTICA", "TECNOLOGIA"},
    "AMAMBAY": {"AGRONOMIA", "COMERCIO", "LOGISTICA"},
    "ASUNCION": {"ADMINISTRACION", "DERECHO", "EDUCACION", "SALUD", "TECNOLOGIA"},
    "BOQUERON": {"AGROINDUSTRIA", "AMBIENTE", "GANADERIA", "LOGISTICA"},
    "CAAGUAZU": {"AGROINDUSTRIA", "AGRONOMIA", "LOGISTICA", "VETERINARIA"},
    "CAAZAPA": {"AGRONOMIA", "AMBIENTE", "EDUCACION", "FORESTAL"},
    "CANINDEYU": {"AGRONOMIA", "AMBIENTE", "LOGISTICA", "VETERINARIA"},
    "CENTRAL": {"ADMINISTRACION", "INDUSTRIA", "SALUD", "TECNOLOGIA"},
    "CONCEPCION": {"AMBIENTE", "FORESTAL", "INDUSTRIA", "INGENIERIA", "LOGISTICA"},
    "CORDILLERA": {"ADMINISTRACION", "TECNOLOGIA", "TURISMO"},
    "GUAIRA": {"AGRONOMIA", "AMBIENTE", "INDUSTRIA", "TURISMO"},
    "ITAPUA": {"AGRONOMIA", "COMERCIO", "SALUD", "TECNOLOGIA"},
    "MISIONES": {"AGROINDUSTRIA", "GANADERIA", "VETERINARIA"},
    "NEEMBUCU": {"AGROINDUSTRIA", "AMBIENTE", "TURISMO"},
    "PARAGUARI": {"ADMINISTRACION", "AGROINDUSTRIA", "TURISMO"},
    "PRESIDENTE HAYES": {"AMBIENTE", "GANADERIA", "LOGISTICA", "VETERINARIA"},
    "SAN PEDRO": {"AGROINDUSTRIA", "AGRONOMIA", "AMBIENTE", "VETERINARIA"},
}

# Mapeo de la clasificacion de area estrategica hacia los ejes productivos
# que esa area satisface. Es un mapeo muchos-a-muchos deliberado: una misma
# area puede cubrir mas de un eje cuando hay solapamiento real de
# competencias (p. ej. "INGENIERIA" satisface tambien INDUSTRIA y
# LOGISTICA; "VETERINARIA" satisface tambien GANADERIA; "FORESTAL"
# satisface tambien AMBIENTE; "ADMINISTRACION" satisface tambien COMERCIO).
MAPEO_AREA_EJE = {
    "AGRONOMIA":      ["agronomia"],
    "AGROINDUSTRIA":  ["agroindustria"],
    "VETERINARIA":    ["veterinaria"],
    "TECNOLOGIA":     ["tecnologia"],
    "INGENIERIA":     ["ingenieria"],
    "INDUSTRIA":      ["industria", "ingenieria"],
    "SALUD":          ["salud"],
    "ADMINISTRACION": ["administracion"],
    "COMERCIO":       ["comercio", "administracion"],
    "LOGISTICA":      ["logistica", "ingenieria"],
    "AMBIENTE":       ["ambiente", "forestal"],
    "FORESTAL":       ["forestal"],
    "TURISMO":        ["turismo"],
    "DERECHO":        ["derecho"],
    "EDUCACION":      ["educacion"],
    "GANADERIA":      ["ganaderia", "veterinaria"],
}

# Seis areas Frascati usadas en la tesis.
AREAS_FRASCATI = {
    "Ciencias Sociales": [
        "ADMIN", "DERECH", "ECON", "CONTAB", "CIENC SOC", "COMUNIC",
        "PEDAG", "EDUC", "TURIS", "COMERC", "MARKETING", "FINAN",
    ],
    "Ingenieria y Tecnologia": [
        "INGEN", "TECNOL", "INFORMAT", "SISTEM", "ELECTR", "CIVIL",
        "MECATR", "TELECOM", "SOFTWARE", "COMPUT", "DATA",
    ],
    "Ciencias Medicas y Salud": [
        "MEDIC", "ENFERMER", "ODONT", "FARMAC", "NUTRIC", "PSICOL",
        "KINES", "SALUD", "FISIOTER",
    ],
    "Humanidades": [
        "LETRAS", "HISTORIA", "FILOSOF", "ARTE", "IDIOM", "TEOLOG", "HUMANID",
    ],
    "Ciencias Naturales": [
        "BIOLOG", "QUIM", "FISIC", "MATEM", "ESTAD", "CIENC NAT", "AMBIENT", "FOREST",
    ],
    "Ciencias Agricolas": [
        "AGRO", "VETERIN", "ZOOTEC", "GANADER", "AGRIC", "PECUAR",
    ],
}

print(f"Configurado para {len(DEPARTAMENTOS)} departamentos/territorios.")

# Poblacion residente de 15 a 29 anos observada en el Censo 2022 del INE.
# Se usa como denominador del IC del ano base. En esta investigacion se denomina
# "demanda potencial" exclusivamente al universo demografico residente 15-29;
# no representa matricula, postulantes, demanda efectiva, cupos ni capacidad.
POBLACION_JOVEN_CENSO_2022 = {
    "ASUNCION": 103630, "CONCEPCION": 51340, "SAN PEDRO": 86568,
    "CORDILLERA": 66395, "GUAIRA": 42398, "CAAGUAZU": 107197,
    "CAAZAPA": 32527, "ITAPUA": 113044, "MISIONES": 27154,
    "PARAGUARI": 44984, "ALTO PARANA": 204454, "CENTRAL": 495995,
    "NEEMBUCU": 16145, "AMAMBAY": 47769, "CANINDEYU": 51743,
    "PRESIDENTE HAYES": 32994, "BOQUERON": 19313, "ALTO PARAGUAY": 4373,
}


def poblacion_joven_2022_confiable(dpto: str) -> float:
    """Devuelve la poblacion residente de 15-29 anos del Censo 2022."""
    return float(POBLACION_JOVEN_CENSO_2022[dpto])


## 4. Funciones utilitarias

Estas funciones se usan transversalmente en todo el pipeline, como soporte
de las funciones de carga, clasificación y cálculo que vienen después:

- **`crear_carpetas`**: recorre la lista `[CARPETA_SALIDA, CARPETA_FIGURAS,
  CARPETA_MAPAS]` y crea cada carpeta si todavía no existe
  (`mkdir(parents=True, exist_ok=True)`), de modo que las celdas posteriores
  puedan guardar archivos sin error.

- **`quitar_tildes`**: recibe un texto y le quita los acentos, aplicando una
  normalización Unicode (NFD) que separa cada letra de su marca diacrítica
  y luego descarta esas marcas (categoría Unicode "Mn").

- **`normalizar_departamento`**: estandariza el nombre de un departamento
  proveniente de cualquiera de las tres fuentes (CONES, INE, GeoJSON), que
  pueden escribirlo de formas distintas. El proceso, en orden: pasa el
  texto a mayúsculas; corrige errores de codificación frecuentes en tildes
  y eñes; quita los acentos con `quitar_tildes`; reemplaza puntos, guiones
  y guiones bajos por espacios; elimina palabras genéricas como
  "DEPARTAMENTO" o "DISTRITO"; interpreta la palabra "CAPITAL" como
  Asunción; quita códigos numéricos o números romanos al inicio del texto;
  y finalmente compara el resultado contra una lista de variantes conocidas
  (por ejemplo, "PDTE HAYES" o "NEMBUCU") para devolver siempre el mismo
  nombre estándar de departamento.

- **`encontrar_columna`**: dado un DataFrame y una lista de nombres de
  columna candidatos, busca cuál de esos nombres existe realmente entre las
  columnas del archivo (sin distinguir mayúsculas/minúsculas ni espacios),
  para tolerar que distintos archivos usen encabezados ligeramente
  distintos para el mismo dato.

- **`convertir_numero`**: convierte un valor de texto a `float`, resolviendo
  la ambigüedad entre el punto y la coma como separador decimal o de miles
  (por ejemplo, "1.884.288" se interpreta como formato paraguayo con puntos
  de miles, mientras que un valor con coma como último separador se
  interpreta como decimal).

- **`minmax`**: normaliza una serie numérica al rango 0-1, restando el
  mínimo y dividiendo por el rango (máximo menos mínimo); si todos los
  valores son iguales, devuelve una serie de ceros para evitar una división
  por cero.

- **`validar_departamentos`**: compara el conjunto de departamentos
  presentes en un DataFrame contra la lista esperada de 18 territorios; si
  falta alguno, detiene la ejecución con un error; si aparece algún
  territorio no reconocido, solo emite una advertencia y continúa.

- **`asegurar_columnas_no_nulas`**: recorre una lista de columnas y, para
  cada una, primero verifica que exista en el DataFrame y luego que no
  tenga valores nulos; si encuentra un nulo, detiene la ejecución indicando
  qué departamentos están afectados.


In [ ]:
def crear_carpetas() -> None:
    for carpeta in [CARPETA_SALIDA, CARPETA_FIGURAS, CARPETA_MAPAS]:
        carpeta.mkdir(parents=True, exist_ok=True)


def quitar_tildes(texto: str) -> str:
    return "".join(
        c for c in unicodedata.normalize("NFD", texto)
        if unicodedata.category(c) != "Mn"
    )


def normalizar_departamento(nombre: object) -> str:
    """
    Normaliza nombres departamentales provenientes del CONES, INE y GeoJSON.
    """
    if pd.isna(nombre):
        return ""

    s = str(nombre).strip().upper()

    # Corregir errores frecuentes de codificacion antes de quitar tildes.
    reemplazos_codificacion = {
        "\u00c3\u2018": "\u00d1",  # Ã‘ -> Ñ
        "\u00c3\u0161": "\u00da",  # Ãš -> Ú
        "\u00c3\u201c": "\u00d3",  # Ã“ -> Ó
        "\u00c3\u2030": "\u00c9",  # Ã‰ -> É
        "\u00c2": "",              # Â  -> (eliminar)
    }
    for original, reemplazo in reemplazos_codificacion.items():
        s = s.replace(original, reemplazo)

    s = quitar_tildes(s)

    s = s.replace(".", " ")
    s = s.replace("-", " ")
    s = s.replace("_", " ")
    s = re.sub(r"\s+", " ", s).strip()

    s = re.sub(r"\bDEPARTAMENTO\b", " ", s)
    s = re.sub(r"\bDISTRITO DE\b", " ", s)
    s = re.sub(r"\bDISTRITO\b", " ", s)
    s = re.sub(r"\bCAPITAL\b", " ASUNCION ", s)

    s = re.sub(r"^\s*\d+\s*", "", s)

    s = re.sub(
        r"^\s*(XVIII|XVII|XVI|XV|XIV|XIII|XII|XI|X|IX|VIII|VII|VI|V|IV|III|II|I)\s+",
        "",
        s,
    )

    s = re.sub(r"\s+", " ", s).strip()

    equivalencias_contenidas = [
        (["PRESIDENTE HAYES", "PDTE HAYES", "PTE HAYES"], "PRESIDENTE HAYES"),
        (["ALTO PARAGUAY"], "ALTO PARAGUAY"),
        (["ALTO PARANA"], "ALTO PARANA"),
        (["SAN PEDRO"], "SAN PEDRO"),
        (["NEEMBUCU", "NEMBUCU", "EEMBUCU"], "NEEMBUCU"),
        (["CANINDEYU"], "CANINDEYU"),
        (["CONCEPCION"], "CONCEPCION"),
        (["CORDILLERA"], "CORDILLERA"),
        (["CAAGUAZU"], "CAAGUAZU"),
        (["CAAZAPA"], "CAAZAPA"),
        (["PARAGUARI"], "PARAGUARI"),
        (["BOQUERON"], "BOQUERON"),
        (["ASUNCION"], "ASUNCION"),
        (["GUAIRA"], "GUAIRA"),
        (["ITAPUA"], "ITAPUA"),
        (["MISIONES"], "MISIONES"),
        (["CENTRAL"], "CENTRAL"),
        (["AMAMBAY"], "AMAMBAY"),
    ]

    for variantes, nombre_estandar in equivalencias_contenidas:
        if any(variante in s for variante in variantes):
            return nombre_estandar

    return s


def encontrar_columna(df: pd.DataFrame, candidatas: Sequence[str]) -> Optional[str]:
    mapa = {str(c).strip().lower(): c for c in df.columns}
    for candidata in candidatas:
        encontrada = mapa.get(candidata.strip().lower())
        if encontrada is not None:
            return encontrada
    return None


def convertir_numero(valor: object) -> float:
    if pd.isna(valor):
        return np.nan
    if isinstance(valor, (int, float, np.integer, np.floating)):
        return float(valor)
    s = str(valor).strip().replace("\xa0", "")
    s = re.sub(r"[^0-9,.-]", "", s)
    if not s:
        return np.nan
    if s.count(".") > 1 and "," not in s:
        s = s.replace(".", "")
    elif s.count(",") > 1 and "." not in s:
        s = s.replace(",", "")
    elif "." in s and "," in s:
        if s.rfind(",") > s.rfind("."):
            s = s.replace(".", "").replace(",", ".")
        else:
            s = s.replace(",", "")
    elif "," in s:
        partes = s.split(",")
        if len(partes[-1]) == 3:
            s = s.replace(",", "")
        else:
            s = s.replace(",", ".")
    elif "." in s:
        partes = s.split(".")
        if len(partes[-1]) == 3:
            s = s.replace(".", "")
    try:
        return float(s)
    except ValueError:
        return np.nan


def minmax(serie: pd.Series) -> pd.Series:
    s = pd.to_numeric(serie, errors="coerce")
    minimo, maximo = s.min(), s.max()
    if pd.isna(minimo) or pd.isna(maximo) or maximo == minimo:
        return pd.Series(0.0, index=serie.index)
    return (s - minimo) / (maximo - minimo)


def validar_departamentos(df: pd.DataFrame, nombre_df: str) -> None:
    presentes = set(df["DPTO_NORM"].dropna().astype(str))
    esperados = set(DEPARTAMENTOS)
    faltantes = sorted(esperados - presentes)
    extras = sorted(presentes - esperados)
    if faltantes:
        raise ValueError(f"{nombre_df}: faltan departamentos/territorios: {faltantes}")
    if extras:
        print(f"ADVERTENCIA - {nombre_df}: se ignoraran territorios no esperados: {extras}")


def asegurar_columnas_no_nulas(df: pd.DataFrame, columnas: Sequence[str], nombre_df: str) -> None:
    for columna in columnas:
        if columna not in df.columns:
            raise ValueError(f"{nombre_df}: no existe la columna requerida '{columna}'.")
        if df[columna].isna().any():
            dptos = df.loc[df[columna].isna(), "DPTO_NORM"].tolist() if "DPTO_NORM" in df else []
            raise ValueError(f"{nombre_df}: valores nulos en '{columna}'. Departamentos: {dptos}")

print("Funciones utilitarias definidas.")


## 5. Carga de los archivos de entrada

El pipeline necesita cuatro archivos:

1. Registro de carreras activas del CONES.
2. Cuadro 1 del Censo 2022, con población total departamental.
3. GeoJSON departamental.
4. **Estimaciones y Proyecciones Departamentales. Revisión 2025** del INE.

El cuarto archivo contiene la serie oficial 2000–2035. El análisis toma los
años 2022–2032 y suma los grupos 15–19, 20–24 y 25–29. No se ajusta ningún
modelo de proyección propio.


In [ ]:
from google.colab import files

print(
    "Selecciona los 4 archivos de entrada: CONES, Cuadro 1 del Censo 2022, "
    "GeoJSON y Proyecciones Departamentales del INE (Revisión 2025)."
)
subidos = files.upload()
print("\nArchivos subidos:")
for nombre in subidos:
    print(f"  - {nombre}")


## 6. Carga de datos base

- `cargar_cones` lee el registro JSON del CONES.
- `cargar_ine_cuadro1` obtiene la población total del Censo 2022 y valida los
  18 territorios.
- `cargar_geojson` normaliza las geometrías departamentales para mapas y Moran.

La carga específica de las proyecciones departamentales se define en la
sección 11, junto con el análisis de brechas. Ya no se calcula una proporción
nacional uniforme de población joven.


In [ ]:
def cargar_cones(ruta: str) -> pd.DataFrame:
    with open(ruta, "r", encoding="utf-8-sig") as archivo:
        datos = json.load(archivo)
    df = pd.DataFrame(datos)
    if df.empty:
        raise ValueError("El archivo CONES esta vacio.")
    return df


def cargar_ine_cuadro1(ruta: str) -> pd.DataFrame:
    """
    Lee el Cuadro 1 del INE segun su estructura real:

    - Columna B: Departamento y distrito.
    - Columna C: Poblacion total.
    - Las filas departamentales comienzan con "Departamento".
    - Asuncion aparece como una fila independiente, sin el prefijo
      "Departamento".

    Se excluyen las filas distritales para evitar confundir el distrito con el
    total del departamento.
    """
    raw = pd.read_excel(ruta, header=None)

    if raw.empty:
        raise ValueError("El archivo INE Cuadro 1 esta vacio.")

    # En Excel: B = indice 1 y C = indice 2.
    nombres = raw.iloc[:, 1].astype(str).str.strip()
    totales = raw.iloc[:, 2].apply(convertir_numero)

    registros: Dict[str, float] = {}

    for nombre, total in zip(nombres, totales):
        if pd.isna(total):
            continue

        nombre_sin_tildes = quitar_tildes(nombre.upper()).strip()

        es_departamento = nombre_sin_tildes.startswith("DEPARTAMENTO ")
        es_asuncion = nombre_sin_tildes == "ASUNCION"

        if not es_departamento and not es_asuncion:
            continue

        dpto = normalizar_departamento(nombre)

        if dpto in DEPARTAMENTOS:
            registros[dpto] = float(total)

    # Correccion explicita documentada en la tesis.
    registros["ASUNCION"] = 462241

    df = pd.DataFrame({
        "DPTO_NORM": DEPARTAMENTOS,
        "POB_TOTAL": [registros.get(d, np.nan) for d in DEPARTAMENTOS],
    })

    validar_departamentos(df, "INE Cuadro 1")
    asegurar_columnas_no_nulas(df, ["POB_TOTAL"], "INE Cuadro 1")

    df["POB_TOTAL"] = df["POB_TOTAL"].round().astype(int)

    print("\nPoblacion total leida desde Cuadro 1:")
    print(df.to_string(index=False))

    return df


def cargar_geojson(ruta: str):
    """
    Carga y normaliza el GeoJSON departamental.
    """
    if gpd is None:
        print(
            "ADVERTENCIA - geopandas no esta instalado; "
            "se omiten analisis y mapas geograficos."
        )
        return None

    try:
        gdf = gpd.read_file(ruta)
    except Exception:
        with open(ruta, "r", encoding="utf-8-sig") as archivo:
            geo = json.load(archivo)
        gdf = gpd.GeoDataFrame.from_features(
            geo["features"],
            crs="EPSG:4326",
        )

    candidatas = [
        "DPTO_DESC",
        "DPTO",
        "DEPARTAMENTO",
        "departamento",
        "NAME_1",
        "name",
        "NOMBRE",
        "DPT_DESC",
        "DPT",
    ]
    col_nombre = encontrar_columna(gdf, candidatas)

    if col_nombre is None:
        raise ValueError(
            "No se encontro columna departamental en el GeoJSON. "
            f"Columnas disponibles: {list(gdf.columns)}"
        )

    print("\nNombres originales encontrados en el GeoJSON:")
    for nombre in sorted(gdf[col_nombre].dropna().astype(str).unique()):
        print(f"   - {nombre}")

    gdf["DPTO_NORM"] = gdf[col_nombre].apply(normalizar_departamento)

    print("\nCorrespondencia de nombres normalizados del GeoJSON:")
    print(
        gdf[[col_nombre, "DPTO_NORM"]]
        .drop_duplicates()
        .sort_values("DPTO_NORM")
        .to_string(index=False)
    )

    no_reconocidos = sorted(
        set(gdf["DPTO_NORM"].dropna().astype(str)) - set(DEPARTAMENTOS)
    )
    if no_reconocidos:
        print(
            "\nADVERTENCIA - nombres territoriales no reconocidos "
            f"y que seran ignorados: {no_reconocidos}"
        )

    gdf = gdf[gdf["DPTO_NORM"].isin(DEPARTAMENTOS)].copy()

    # Unir geometrias repetidas correspondientes al mismo departamento.
    gdf = gdf.dissolve(by="DPTO_NORM", as_index=False)

    validar_departamentos(gdf, "GeoJSON")

    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326")

    return gdf

print("Funciones de carga de datos base definidas.")


## 7. Preparación y clasificación de la oferta (CONES)

Esta es la sección más extensa del pipeline, ya que clasifica cada carrera
registrada en tres dimensiones distintas antes de que la oferta pueda
contabilizarse por departamento:

- **`normalizar_modalidad`**: recibe el texto de modalidad declarado en el
  registro CONES, lo pasa a mayúsculas y le quita los acentos, y luego
  evalúa un conjunto de palabras clave en un orden específico: primero
  busca indicios de semipresencialidad o educación a distancia ("SEMI",
  "HIBRID", "MEDIAD", "DISTANCIA"); si no encuentra ninguno, busca indicios
  de modalidad virtual ("VIRT", "ONLINE", "EN LINEA"); si tampoco encuentra
  eso, busca "PRES" para clasificar como Presencial; y si ninguna de las
  anteriores aplica, la modalidad queda como "OTRA". El orden importa
  porque evita que una carrera semipresencial, cuyo texto también podría
  contener la palabra "presencial", termine clasificada como puramente
  Presencial.

- **`clasificar_frascati`**: recorre el diccionario `AREAS_FRASCATI`
  (definido en la sección de Configuración) y, para cada una de las 6
  áreas, verifica si alguna de sus palabras clave aparece dentro del texto
  de la carrera (sin tildes, en mayúsculas). Devuelve el nombre de la
  primera área cuya palabra clave coincide, o `None` si ninguna coincide.
  Esta clasificación amplia alimenta más adelante el índice de diversidad
  disciplinar (HHI).

- **`clasificar_area_estrategica`**: aplica una cadena de reglas
  `if/elif` mucho más fina, con 16 categorías posibles (Agroindustria,
  Ganadería, Agronomía, Veterinaria, Tecnología, Ingeniería, Salud,
  Administración, Comercio, Logística, Forestal, Ambiente, Turismo,
  Derecho, Educación, y un catch-all genérico de Ingeniería), devolviendo
  "SIN_CLASIFICAR" si ninguna regla coincide. El orden de evaluación es
  deliberado: por ejemplo, la regla de Ganadería se evalúa antes que
  Agronomía, Ingeniería y Administración, porque denominaciones reales
  como "Ingeniería en Producción Agropecuaria" contienen palabras genéricas
  ("INGEN", "ADMIN") que, evaluadas primero, clasificarían mal la carrera
  antes de llegar a la regla más específica de Ganadería. Por el mismo
  motivo, el catch-all genérico de "INGEN" se ubica al final de toda la
  cadena, para que casos como "Ingeniería Ambiental" o "Ingeniería
  Comercial" se resuelvan primero en sus áreas específicas (Ambiente,
  Administración) antes de caer en Ingeniería genérica.

- **`mapear_area_a_ejes`**: toma el resultado de
  `clasificar_area_estrategica` para una carrera y devuelve la lista de
  ejes productivos (definidos en `MAPEO_AREA_EJE`) que esa área satisface;
  si el área es "SIN_CLASIFICAR" o nula, devuelve una lista vacía.

- **`preparar_cones`**: es la función principal de esta sección. Localiza
  las columnas relevantes del archivo crudo (departamento, modalidad,
  estado activo/inactivo, nombre de carrera y área oficial, si existe)
  usando `encontrar_columna`; normaliza el nombre del departamento y
  descarta las filas cuyo departamento no está entre los 18 esperados;
  si existe una columna de estado, filtra solo las carreras marcadas como
  activas; elimina filas con nombre de carrera o departamento nulos (sin
  eliminar registros duplicados, ya que cada fila representa una oferta
  registrada distinta, aunque el nombre de la carrera se repita entre
  instituciones); y finalmente aplica las tres clasificaciones descriptas
  arriba a cada carrera, guardando los resultados en las columnas
  `MODALIDAD`, `AREA_FRASCATI`, `AREA_ESTRATEGICA` y `EJES_ESTRATEGICOS`.
  Al terminar, imprime la cantidad de registros procesados y cuántos
  quedaron sin clasificar en cada dimensión.


In [ ]:
def normalizar_modalidad(valor: object) -> str:
    s = quitar_tildes("" if pd.isna(valor) else str(valor).upper().strip())
    # El orden evita que SEMIPRESENCIAL sea contado tambien como PRESENCIAL.
    if any(k in s for k in ["SEMI", "HIBRID", "MEDIAD", "DISTANCIA"]):
        return "SEMIPRESENCIAL_DISTANCIA"
    if "VIRT" in s or "ONLINE" in s or "EN LINEA" in s:
        return "VIRTUAL"
    if "PRES" in s:
        return "PRESENCIAL"
    return "OTRA"



def clasificar_frascati(nombre: object) -> Optional[str]:
    s = quitar_tildes("" if pd.isna(nombre) else str(nombre).upper())
    for area, palabras in AREAS_FRASCATI.items():
        if any(palabra in s for palabra in palabras):
            return area
    return None


def clasificar_area_estrategica(area: object) -> str:
    """
    Clasifica la denominacion de una carrera en una de 16 areas estrategicas
    de base (o "SIN_CLASIFICAR" si ninguna regla matchea). Las reglas estan
    ordenadas para evitar coincidencias parciales indeseadas (por ejemplo,
    Agroindustria se evalua antes que Agronomia).
    """
    s = quitar_tildes("" if pd.isna(area) else str(area).upper())

    # Agroindustria (antes que Agronomia)
    if any(k in s for k in ["AGROIND", "AGROALIM", "INDUSTRIA ALIM",
                             "TECNOLOGIA ALIM", "CIENCIA Y TECNOLOGIA DE ALIM",
                             "TECNOLOGIA DE LA PRODUCCION ALIM",
                             "CIENCIA Y TECNOLOGIA DE ALIMENTOS"]):
        return "AGROINDUSTRIA"

    # Ganaderia (se evalua deliberadamente ANTES que Agronomia, Ingenieria y
    # Administracion: denominaciones reales como "Ingenieria en Produccion
    # Agropecuaria" o "Administracion Agropecuaria" contienen palabras
    # genericas como "INGEN" o "ADMIN" que, de evaluarse primero,
    # secuestrarian la clasificacion antes de llegar a esta regla mas
    # especifica.
    if any(k in s for k in ["GANAD", "PECUAR", "AGROPEC", "PRODUCCION GANAD",
                             "BOVINO", "PORCINO", "OVINO", "EQUINO"]):
        return "GANADERIA"

    # Agronomia (se excluye 'AGROPEC'/'PRODUCCION AGROPEC' porque ya quedo
    # cubierto arriba por la regla de Ganaderia).
    if any(k in s for k in ["AGRON", "AGRIC",
                             "PRODUCCION VEGETAL", "FITOSANIDAD", "PROTECCION VEGETAL",
                             "PRODUCCION AGROEC", "NEGOCIOS AGROPEC", "GESTION AGROPEC",
                             "ADMINISTRACION AGROPEC", "ADMINISTRACION AGRARIA",
                             "ADMINISTRACION RURAL"]):
        return "AGRONOMIA"

    # Veterinaria / Sanidad animal
    if any(k in s for k in ["VETERIN", "CIENCIAS VETERINARIA", "MEDICINA VETERINARIA",
                             "SANIDAD ANIMAL", "ZOOTECNIA", "PRODUCCION ANIMAL"]):
        return "VETERINARIA"

    # Tecnologia / Informatica / Sistemas
    if any(k in s for k in ["INFORM", "SISTEM", "SOFTW", "COMPUT", "PROGRAMAC",
                             "DATA", "CIBERSEGUR", "ROBOTICA", "AUTOMATIZAC",
                             "MECATRONIC", "ELECTRONICA", "ELECTROTECNIA",
                             "DESARROLLO WEB", "DESARROLLO DE SOFTWARE", "REDES",
                             "INTELIGENCIA ARTIFICIAL", "ANALISIS DE SISTEM",
                             "CIENCIAS INFORMATICA", "CIENCIAS DE LA COMPUTAC",
                             "CIENCIAS DE LA INFORMAC", "TECNOLOG",
                             "VIDEOJUEGO", "APLICACIONES PARA DISPOSITIVOS"]) or re.search(r"\bTIC\b", s):
        return "TECNOLOGIA"

    # Ingenieria: sub-especialidades especificas. Deliberadamente SIN el
    # catch-all generico "INGEN" aqui, para que "Ingenieria Ambiental",
    # "Ingenieria Comercial" o "Ingenieria Forestal" se clasifiquen primero
    # en su area especifica.
    if any(k in s for k in ["ELECTROMECANIC", "ELECTRIC",
                             "MECANICA", "MECANIC", "AUTOMOTRIZ", "REFRIGER",
                             "CLIMATIZAC", "BIOMEDIC", "QUIMICA INDUSTRI",
                             "CONSTRUCCION NAVAL", "NAVAL", "AERONAUTIC", "AERONAV",
                             "VIAL", "HIDRO", "GEODESIC", "TOPOGRAF",
                             "ARQUITECTURA", "INDUSTRI"]):
        return "INGENIERIA"

    # Salud
    if any(k in s for k in ["MEDIC", "ENFERMER", "SALUD", "ODONT", "FARMAC",
                             "NUTRICION", "NUTRIC", "KINES", "FISIOTER",
                             "LABORATORIO CLINIC", "RADIOLOG", "PROTESIS",
                             "QUIRURG", "OBSTETRIC", "EMERGENCIA",
                             "MASAJE TERAP", "BIOQUIMIC", "TERAPIA", "REHABILITAC",
                             "PODOLOG", "OPTIC", "COSMET", "COSMIATR", "ESTETICA", "FONOAUD",
                             "PSICOLOG", "PSIQUIATR", "SALUD PUBLICA", "SALUD MENTAL",
                             "GERIATRI", "NEONATOL", "EPIDEMIOL", "IMAGENOL",
                             "HEMATER", "VISITADOR MEDICO", "CIENCIAS DEL LABORATORIO",
                             "CIENCIAS MEDIC", "CIENCIAS DE LA SALUD",
                             "PATOLOG", "CARDIOLOG", "GASTROENTEROLOG", "HEPATOLOG",
                             "NEFROLOG", "NEUROCIENC", "ENDOSCOP", "BACTERIOLOG",
                             "PERINATOLOG", "OSTEOPAT", "HEMODIALISIS", "DIALISIS",
                             "ORTODONC", "ORTOPEDIA"]):
        return "SALUD"

    # Administracion / Contabilidad / Finanzas / Economia
    if any(k in s for k in ["ADMIN", "CONTAB", "CONTAD", "FINAN", "ECON",
                             "NEGOC", "AUDITOR", "TRIBUTAC", "IMPUEST", "COMERCIAL",
                             "CIENCIAS CONTABLES", "CIENCIAS ECONOMICA",
                             "COOPERATIV", "COOPERATISM", "BANCARI", "SEGUROS",
                             "GESTION EMPRESARIAL", "GESTION DE EMPRESAS",
                             "GESTION DE NEGOCIOS", "GESTION CONTABLE",
                             "RECURSOS HUMANOS", "GESTION DE RECURSOS HUMANOS",
                             "ADMINISTRACION DE NEGOCIOS", "ADMINISTRACION DE EMPRESAS",
                             "ADMINISTRACION HOSPITALARIA", "ADMINISTRACION PUBLICA"]):
        return "ADMINISTRACION"

    # Comercio / Marketing / Ventas
    if any(k in s for k in ["COMERC", "EXPORT", "IMPORT", "MARKETING", "PUBLICIDAD",
                             "VENTAS", "MERCADOTECNIA", "COMERCIALIZAC",
                             "RELACIONES PUBLICAS", "NEGOCIOS INTERNACIONALES",
                             "COMERCIO INTERNAC"]):
        return "COMERCIO"

    # Logistica / Transporte / Aduana
    if any(k in s for k in ["LOGIST", "TRANSP", "ADUANER", "ADUANAL",
                             "COMERCIO EXTERIOR", "GESTION ADUANERA",
                             "ADMINISTRACION ADUANERA"]):
        return "LOGISTICA"

    # Forestal
    if any(k in s for k in ["FOREST", "FORESTAL", "MADERA", "AGROFOREST"]):
        return "FORESTAL"

    # Medio ambiente / Ecologia
    if any(k in s for k in ["AMBIE", "ECOLOG", "MEDIO AMBIENTE", "RECURSOS NAT",
                             "GESTION AMBIENTAL", "CIENCIAS AMBIENTALES",
                             "INGENIERIA AMBIENTAL", "GESTION Y AUDITORIA AMBIENTAL",
                             "PRODUCCION AGROECOL", "AGROECOL"]):
        return "AMBIENTE"

    # Turismo / Hoteleria / Gastronomia
    if any(k in s for k in ["TURIS", "HOTEL", "GASTRON", "HOTELERIA",
                             "GESTION DE HOSPITALIDAD", "ALTA COCINA", "PASTELERIA",
                             "GESTION DE ALOJAMIENTO"]):
        return "TURISMO"

    # Derecho / Notariado / Criminalistica
    if any(k in s for k in ["DERECH", "JURID", "NOTARI", "ESCRIBANIA", "ABOGAC",
                             "CRIMIN", "FORENSE", "PENAL",
                             "SEGURIDAD PUBLICA", "POLICIAL", "PENITENCIARI",
                             "LEGISLAT"]):
        return "DERECHO"

    # Educacion / Pedagogia / Docencia
    if any(k in s for k in ["EDUC", "DOCEN", "PEDAG", "PROFESORADO",
                             "HABILITAC PEDAG", "HABILITACION PEDAGOGICA",
                             "CIENCIAS DE LA EDUCAC", "DIDACTIC",
                             "PRIMERA INFANCIA", "EDUCACION INICIAL",
                             "EDUCACION ESCOLAR", "TECNICO DOCENTE"]):
        return "EDUCACION"

    # Ingenieria generica (catch-all): se evalua deliberadamente al final de
    # toda la cadena. Solo si ninguna regla anterior (mas especifica)
    # matcheo, una carrera que contenga "Ingenieria" cae aqui como
    # INGENIERIA generica (p. ej. "Ingenieria Civil").
    if "INGEN" in s:
        return "INGENIERIA"

    return "SIN_CLASIFICAR"


def mapear_area_a_ejes(area_clasificada: object) -> List[str]:
    """
    Traduce una categoria de area estrategica a la lista de ejes productivos
    que esa area satisface, segun MAPEO_AREA_EJE.
    """
    if pd.isna(area_clasificada):
        return []
    a = str(area_clasificada).strip().lower()
    if a in ("", "sin_clasificar"):
        return []
    return [eje for eje, palabras_area in MAPEO_AREA_EJE.items() if a in palabras_area]


def preparar_cones(df_raw: pd.DataFrame) -> pd.DataFrame:
    col_depto = encontrar_columna(df_raw, ["departamento", "Departamento"])
    col_modalidad = encontrar_columna(df_raw, ["modalidad_estudio", "modalidad", "Modalidad"])
    col_activa = encontrar_columna(df_raw, ["activa", "Activa", "estado"])
    col_nombre = encontrar_columna(df_raw, [
        "denominacion_carrera",
        "nombre_carrera",
        "carrera",
        "denominacion",
        "programa",
        "titulo_carrera",
    ])
    col_area = encontrar_columna(df_raw, [
        "clasificacion_campo_amplio", "clasificacion_campo", "area", "Area", "campo_amplio",
    ])

    if col_depto is None or col_nombre is None:
        raise ValueError(
            "No se identificaron las columnas requeridas del CONES. "
            f"Departamento detectado: {col_depto}; "
            f"carrera detectada: {col_nombre}. "
            f"Columnas disponibles: {list(df_raw.columns)}"
        )

    df = df_raw.copy()
    df["DPTO_NORM"] = df[col_depto].apply(normalizar_departamento)
    df = df[df["DPTO_NORM"].isin(DEPARTAMENTOS)].copy()

    if col_activa is not None:
        estados = df[col_activa].fillna("").astype(str).str.strip().str.upper().apply(quitar_tildes)
        df = df[estados.isin(["SI", "TRUE", "1", "ACTIVA", "ACTIVO"])].copy()

    # Nulos criticos.
    # IMPORTANTE: no se eliminan duplicados, porque cada fila del registro CONES
    # representa una oferta/carrera registrada.
    df = df.dropna(subset=[col_depto, col_nombre]).copy()

    df["NOMBRE_CARRERA"] = df[col_nombre].astype(str).str.strip()
    df["MODALIDAD"] = (
        df[col_modalidad].apply(normalizar_modalidad)
        if col_modalidad is not None else "OTRA"
    )

    # Se prioriza el campo disciplinar oficial si existe; el nombre sirve de respaldo.
    base_area = (
        df[col_area].fillna("").astype(str) + " " + df["NOMBRE_CARRERA"]
        if col_area is not None else df["NOMBRE_CARRERA"]
    )
    df["AREA_FRASCATI"] = base_area.apply(clasificar_frascati)
    # IMPORTANTE: AREA_ESTRATEGICA usa UNICAMENTE el nombre de la carrera,
    # sin el campo oficial, para no secuestrar la clasificacion fina.
    df["AREA_ESTRATEGICA"] = df["NOMBRE_CARRERA"].apply(clasificar_area_estrategica)
    df["EJES_ESTRATEGICOS"] = df["AREA_ESTRATEGICA"].apply(mapear_area_a_ejes)

    print(f"CONES: {len(df):,} registros de carreras activas validas despues de limpieza (sin eliminar duplicados).")
    print(f"Carreras clasificadas en Frascati: {df['AREA_FRASCATI'].notna().sum():,}")
    print(f"Carreras sin clasificacion Frascati: {df['AREA_FRASCATI'].isna().sum():,}")
    sin_area_estrategica = (df["AREA_ESTRATEGICA"] == "SIN_CLASIFICAR").sum()
    print(f"Carreras sin area estrategica (SIN_CLASIFICAR): {sin_area_estrategica:,}")
    return df

print("Funciones de clasificacion y preparacion de CONES definidas.")


## 8. Oferta, demanda potencial e indicadores

En esta tesis, **demanda potencial** significa únicamente la población
residente de 15 a 29 años que constituye el universo demográfico potencial de
la educación superior. No equivale a matrícula, postulantes, intención de
estudio, demanda efectiva, cupos ni capacidad institucional.

El índice de cobertura (IC) relaciona la cantidad de carreras presenciales
registradas con esa población, por cada 10.000 jóvenes. Por ello mide
**disponibilidad territorial de oferta registrada**, no cobertura de personas.
Asunción se conserva en los cuadros descriptivos, pero se excluye de los
umbrales comparativos y del clustering por su condición de outlier y por la
movilidad interdepartamental hacia la capital.


In [ ]:
def construir_oferta(df_cones: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    base = pd.DataFrame({"DPTO_NORM": DEPARTAMENTOS})

    total = df_cones.groupby("DPTO_NORM").size().rename("C_TOTAL")
    pres = df_cones[df_cones["MODALIDAD"] == "PRESENCIAL"].groupby("DPTO_NORM").size().rename("C_PRES")
    semi = df_cones[df_cones["MODALIDAD"] == "SEMIPRESENCIAL_DISTANCIA"].groupby("DPTO_NORM").size().rename("C_SEMI")
    virt = df_cones[df_cones["MODALIDAD"] == "VIRTUAL"].groupby("DPTO_NORM").size().rename("C_VIRT")
    otra = df_cones[df_cones["MODALIDAD"] == "OTRA"].groupby("DPTO_NORM").size().rename("C_OTRA")

    oferta = base.set_index("DPTO_NORM").join([total, pres, semi, virt, otra]).fillna(0).reset_index()
    for col in ["C_TOTAL", "C_PRES", "C_SEMI", "C_VIRT", "C_OTRA"]:
        oferta[col] = oferta[col].astype(int)

    por_area = (
        df_cones[df_cones["AREA_FRASCATI"].notna()]
        .groupby(["DPTO_NORM", "AREA_FRASCATI"])
        .size()
        .reset_index(name="CARRERAS_AREA")
    )
    return oferta, por_area


def construir_demanda(df_pob: pd.DataFrame) -> pd.DataFrame:
    """Agrega la demanda potencial observada: residentes 15-29 del Censo 2022.

    La variable es un proxy demografico y no una medicion de demanda efectiva.
    """
    df = df_pob.copy()
    df["POB_15_29_EST"] = df["DPTO_NORM"].apply(poblacion_joven_2022_confiable)
    df["DEFINICION_DEMANDA"] = (
        "Poblacion residente de 15 a 29 anos; proxy demografico, no matricula "
        "ni demanda efectiva"
    )
    return df


def construir_indices(df: pd.DataFrame) -> pd.DataFrame:
    salida = df.copy()
    asegurar_columnas_no_nulas(salida, ["POB_15_29_EST"], "DataFrame de indices")
    if (salida["POB_15_29_EST"] <= 0).any():
        raise ValueError("La poblacion joven debe ser mayor que cero.")

    for origen, destino in [
        ("C_TOTAL", "IC_TOTAL"), ("C_PRES", "IC_PRES"),
        ("C_SEMI", "IC_SEMI"), ("C_VIRT", "IC_VIRT"),
    ]:
        salida[destino] = salida[origen] / salida["POB_15_29_EST"] * 10000

    universo = salida["DPTO_NORM"] != DEPARTAMENTO_EXCLUIDO_CLUSTERING
    referencia = salida.loc[universo, "IC_PRES"]
    mediana = float(referencia.median())
    salida["BRECHA_MEDIANA"] = salida["IC_PRES"] - mediana
    q1, q2, q3 = referencia.quantile([0.25, 0.50, 0.75])

    def cuartil(valor: float) -> str:
        if valor < q1:
            return "Q1"
        if valor < q2:
            return "Q2"
        if valor < q3:
            return "Q3"
        return "Q4"

    salida["CUARTIL"] = salida["IC_PRES"].apply(cuartil)
    salida.loc[~universo, "CUARTIL"] = "OUTLIER"
    salida["NIVEL_CUARTIL"] = salida["CUARTIL"].map({
        "Q1": "Cobertura muy baja", "Q2": "Cobertura baja",
        "Q3": "Cobertura media", "Q4": "Cobertura alta",
        "OUTLIER": "Fuera del universo comparativo",
    })
    salida["MEDIANA_IC_17_TERRITORIOS"] = mediana
    return salida


def calcular_tabla_ranking_divergencia(df: pd.DataFrame, umbral_divergencia: int = 8) -> pd.DataFrame:
    """
    Reproduce la Tabla 3.2 de la tesis (oferta absoluta vs. Indice de
    Cobertura Relativa).
    """
    salida = df.copy()
    salida["RANK_ABS"] = salida["C_TOTAL"].rank(method="min", ascending=False).astype(int)
    salida["RANK_IC"] = salida["IC_PRES"].rank(method="min", ascending=False).astype(int)
    salida["DIVERGENCIA_RANKING"] = (salida["RANK_ABS"] - salida["RANK_IC"]).abs()
    salida["DIVERGENCIA_SIGNIFICATIVA"] = salida["DIVERGENCIA_RANKING"] >= umbral_divergencia

    columnas = [
        "DPTO_NORM", "POB_15_29_EST", "C_TOTAL", "C_PRES",
        "RANK_ABS", "RANK_IC", "IC_PRES", "CUARTIL",
        "DIVERGENCIA_RANKING", "DIVERGENCIA_SIGNIFICATIVA",
    ]
    tabla = salida[columnas].sort_values("RANK_ABS").reset_index(drop=True)

    print("\nTABLA 3.2 - OFERTA ABSOLUTA vs. INDICE DE COBERTURA RELATIVA")
    print(
        tabla.assign(
            NOMBRE=lambda d: d["DPTO_NORM"].map(NOMBRES_DISPLAY)
        )[["NOMBRE", "C_TOTAL", "C_PRES", "RANK_ABS", "RANK_IC", "IC_PRES", "CUARTIL", "DIVERGENCIA_SIGNIFICATIVA"]]
        .round({"IC_PRES": 2})
        .to_string(index=False)
    )
    divergentes = tabla.loc[tabla["DIVERGENCIA_SIGNIFICATIVA"], "DPTO_NORM"].tolist()
    print(f"Departamentos con divergencia >= {umbral_divergencia} posiciones: {', '.join(divergentes) if divergentes else 'ninguno'}")

    return tabla


def calcular_diversidad_hhi(df_cones: pd.DataFrame) -> pd.DataFrame:
    clasificadas = df_cones[df_cones["AREA_FRASCATI"].notna()].copy()
    filas = []
    for dpto in DEPARTAMENTOS:
        sub = clasificadas[clasificadas["DPTO_NORM"] == dpto]
        if sub.empty:
            filas.append({
                "DPTO_NORM": dpto,
                "CARRERAS_CLASIFICADAS": 0,
                "AREAS_UNICAS": 0,
                "HHI": np.nan,
                "DIVERSIDAD": 0.0,
            })
            continue
        conteos = sub["AREA_FRASCATI"].value_counts()
        proporciones = conteos / conteos.sum()
        hhi = float((proporciones ** 2).sum())
        filas.append({
            "DPTO_NORM": dpto,
            "CARRERAS_CLASIFICADAS": int(conteos.sum()),
            "AREAS_UNICAS": int(conteos.size),
            "HHI": hhi,
            "DIVERSIDAD": 1.0 - hhi,
        })
    return pd.DataFrame(filas)


def calcular_pertinencia(df_cones: pd.DataFrame) -> pd.DataFrame:
    filas = []
    for dpto in DEPARTAMENTOS:
        requeridas = EJES_PRODUCTIVOS.get(dpto, set())
        sub = df_cones.loc[df_cones["DPTO_NORM"] == dpto, "EJES_ESTRATEGICOS"]
        ofertadas: Set[str] = set()
        for lista_ejes in sub:
            if isinstance(lista_ejes, list):
                ofertadas.update(lista_ejes)
        cubiertas = requeridas & ofertadas
        faltantes = requeridas - ofertadas
        score = len(cubiertas) / len(requeridas) if requeridas else np.nan
        filas.append({
            "DPTO_NORM": dpto,
            "AREAS_REQUERIDAS": ", ".join(sorted(requeridas)),
            "AREAS_CUBIERTAS": ", ".join(sorted(cubiertas)),
            "AREAS_FALTANTES": ", ".join(sorted(faltantes)),
            "SCORE_PERTINENCIA": score,
        })
    return pd.DataFrame(filas)


def exportar_auditoria_clasificacion(df_cones: pd.DataFrame) -> pd.DataFrame:
    """
    Exporta, para cada carrera activa del CONES, su departamento,
    denominacion, modalidad, area Frascati, area estrategica y los ejes
    productivos que termina cubriendo.
    """
    columnas = [
        "DPTO_NORM", "NOMBRE_CARRERA", "MODALIDAD",
        "AREA_FRASCATI", "AREA_ESTRATEGICA", "EJES_ESTRATEGICOS",
    ]
    auditoria = df_cones[columnas].copy()
    auditoria["EJES_ESTRATEGICOS"] = auditoria["EJES_ESTRATEGICOS"].apply(
        lambda lista: ", ".join(lista) if isinstance(lista, list) and lista else ""
    )
    auditoria = auditoria.sort_values(["DPTO_NORM", "AREA_ESTRATEGICA", "NOMBRE_CARRERA"]).reset_index(drop=True)
    auditoria.to_csv(CARPETA_SALIDA / "auditoria_clasificacion_carreras.csv", index=False, encoding="utf-8-sig")

    sin_clasificar = auditoria.loc[auditoria["AREA_ESTRATEGICA"] == "SIN_CLASIFICAR"].copy()
    sin_clasificar.to_csv(CARPETA_SALIDA / "carreras_sin_clasificar.csv", index=False, encoding="utf-8-sig")

    print(f"\nAuditoria de clasificacion exportada ({len(auditoria):,} carreras).")
    print(f"Carreras con AREA_ESTRATEGICA = SIN_CLASIFICAR: {len(sin_clasificar):,}")

    if not sin_clasificar.empty:
        nombres_unicos = (
            sin_clasificar["NOMBRE_CARRERA"]
            .value_counts()
            .reset_index()
        )
        nombres_unicos.columns = ["NOMBRE_CARRERA", "CANTIDAD_REGISTROS"]
        print(f"\nDenominaciones distintas sin clasificar: {len(nombres_unicos):,}")
        print("\nLISTADO DE CARRERAS SIN CLASIFICAR (denominacion unica + cantidad de registros):")
        print(nombres_unicos.to_string(index=False))

    return auditoria


def generar_tabla_4_3(df_general: pd.DataFrame) -> pd.DataFrame:
    """Genera la Tabla 4.3 con el metodo de segmentacion finalmente elegido."""
    requeridas = [
        "DPTO_NORM", "C_TOTAL", "IC_PRES", "SCORE_PERTINENCIA",
        "CLASIFICACION_FINAL", "PRIORIDAD_FINAL", "METODO_SEGMENTACION",
    ]
    faltantes = [c for c in requeridas if c not in df_general.columns]
    if faltantes:
        raise ValueError(f"Faltan columnas para la Tabla 4.3: {faltantes}")

    tabla = df_general[requeridas].copy()
    tabla["Departamento"] = tabla["DPTO_NORM"].map(NOMBRES_DISPLAY)
    tabla.loc[
        tabla["DPTO_NORM"] == DEPARTAMENTO_EXCLUIDO_CLUSTERING,
        ["CLASIFICACION_FINAL", "PRIORIDAD_FINAL", "METODO_SEGMENTACION"],
    ] = ["NO APLICA", "NO APLICA (outlier)", "Excluida"]
    tabla = tabla.rename(columns={
        "C_TOTAL": "Carreras", "IC_PRES": "IC (x10K)",
        "SCORE_PERTINENCIA": "Score Pertinencia",
        "CLASIFICACION_FINAL": "Segmentacion",
        "PRIORIDAD_FINAL": "Prioridad", "METODO_SEGMENTACION": "Metodo",
    })
    tabla = tabla[[
        "Departamento", "Carreras", "IC (x10K)", "Score Pertinencia",
        "Segmentacion", "Prioridad", "Metodo",
    ]].sort_values("IC (x10K)").reset_index(drop=True)
    tabla["IC (x10K)"] = tabla["IC (x10K)"].round(2)
    tabla["Score Pertinencia"] = tabla["Score Pertinencia"].round(3)
    print()
    print("TABLA 4.3 - RESUMEN DE RESULTADOS POR TERRITORIO (BASE 2022)")
    print(tabla.to_string(index=False))
    tabla.to_csv(
        CARPETA_SALIDA / "tabla_4_3_resumen_resultados_departamento.csv",
        index=False, encoding="utf-8-sig",
    )
    return tabla


print("Funciones de oferta, demanda potencial e indicadores definidas.")


## 9. Análisis exploratorio (EDA) e Índice de Moran

`ejecutar_eda` produce estadísticos descriptivos y el Z-score del IC. Los
percentiles y la mediana de referencia se calculan sobre los 17 territorios,
sin Asunción. La matriz de correlación se omite por decisión metodológica.

`calcular_moran` evalúa la autocorrelación espacial del IC con pesos de
contigüidad Reina y 999 permutaciones cuando las dependencias geoespaciales
están disponibles.


In [ ]:
def ejecutar_eda(df: pd.DataFrame) -> pd.DataFrame:
    variables = [
        "POB_TOTAL", "POB_15_29_EST", "C_TOTAL", "C_PRES", "C_SEMI", "C_VIRT",
        "IC_TOTAL", "IC_PRES", "IC_SEMI", "IC_VIRT", "DIVERSIDAD", "SCORE_PERTINENCIA",
    ]
    desc = df[variables].describe().T
    desc["IQR"] = desc["75%"] - desc["25%"]
    desc.to_csv(CARPETA_SALIDA / "estadisticos_descriptivos.csv", encoding="utf-8-sig")

    media = df["IC_PRES"].mean()
    desvio = df["IC_PRES"].std(ddof=1)
    df["Z_IC_PRES"] = (df["IC_PRES"] - media) / desvio if desvio > 0 else 0

    print("\nESTADISTICOS DEL IC PRESENCIAL")
    print(df["IC_PRES"].describe().round(3))
    z_asu = df.loc[df["DPTO_NORM"] == "ASUNCION", "Z_IC_PRES"].iloc[0]
    print(f"Z-score de Asuncion: {z_asu:.3f}")
    universo = df["DPTO_NORM"] != DEPARTAMENTO_EXCLUIDO_CLUSTERING
    referencia = df.loc[universo, "IC_PRES"]
    print(f"Mediana del universo sin Asuncion: {referencia.median():.3f}")
    print(f"p25: {referencia.quantile(.25):.3f}")
    print(f"p75: {referencia.quantile(.75):.3f}")

    pob_q12 = df.loc[df["CUARTIL"].isin(["Q1", "Q2"]), "POB_15_29_EST"].sum()
    pct_q12 = pob_q12 / df.loc[universo, "POB_15_29_EST"].sum() * 100
    print(f"Poblacion joven en Q1 y Q2: {pct_q12:.1f}%")
    return df


def calcular_moran(gdf, df: pd.DataFrame) -> pd.DataFrame:
    if gdf is None or Queen is None or Moran is None:
        print("ADVERTENCIA - no se calculo Moran. Instale geopandas, libpysal y esda.")
        return pd.DataFrame([{
            "VARIABLE": "IC_PRES",
            "MORAN_I": np.nan,
            "P_VALOR_PERMUTACION": np.nan,
            "Z_SIM": np.nan,
            "PERMUTACIONES": 999,
            "NOTA": "No calculado por dependencias faltantes",
        }])

    geo = gdf.merge(df[["DPTO_NORM", "IC_PRES"]], on="DPTO_NORM", how="left")
    geo = geo.sort_values("DPTO_NORM").reset_index(drop=True)
    asegurar_columnas_no_nulas(geo, ["IC_PRES"], "GeoDataFrame para Moran")

    pesos = Queen.from_dataframe(geo, use_index=True)
    pesos.transform = "R"
    moran = Moran(geo["IC_PRES"].to_numpy(), pesos, permutations=999)
    resultado = pd.DataFrame([{
        "VARIABLE": "IC_PRES",
        "MORAN_I": moran.I,
        "P_VALOR_PERMUTACION": moran.p_sim,
        "Z_SIM": moran.z_sim,
        "PERMUTACIONES": 999,
        "NOTA": "Autocorrelacion positiva" if moran.I > 0 else "Autocorrelacion negativa",
    }])
    print("\nINDICE DE MORAN")
    print(resultado.round(4).to_string(index=False))
    return resultado

print("Funciones de EDA e Indice de Moran definidas.")


## 10. Segmentación territorial sin Asunción

Asunción se excluye **antes** del escalamiento y de cualquier cálculo de
segmentación. Sobre los 17 territorios restantes se evalúan K=2,…,8 con:

- método del codo (máxima distancia a la recta entre los extremos), y
- coeficiente de silueta promedio.

La decisión se toma con la silueta. Si su máximo ocurre en K=2, el cuaderno no
retiene K-means y utiliza una clasificación por cuartiles del IC calculada
exclusivamente sobre los 17 territorios. En otro caso, ajusta K-means con el K
seleccionado y contrasta su ordenamiento con HAC Ward.


In [ ]:
def validar_numero_clusters(
    X_scaled: np.ndarray,
) -> Tuple[pd.DataFrame, int, int]:
    if len(X_scaled) < 4:
        raise ValueError("Se requieren al menos cuatro observaciones.")

    k_max = min(K_MAX, len(X_scaled) - 1)
    resultados = []
    for k in range(K_MIN, k_max + 1):
        modelo = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20)
        etiquetas = modelo.fit_predict(X_scaled)
        resultados.append({
            "K": k, "INERCIA": float(modelo.inertia_),
            "SILUETA": float(silhouette_score(X_scaled, etiquetas)),
            "N_TERRITORIOS": len(X_scaled),
            "EXCLUSION": NOMBRES_DISPLAY[DEPARTAMENTO_EXCLUIDO_CLUSTERING],
        })

    validacion = pd.DataFrame(resultados)
    x = validacion["K"].to_numpy(dtype=float)
    y = validacion["INERCIA"].to_numpy(dtype=float)
    x = (x - x.min()) / (x.max() - x.min())
    y = (y - y.min()) / (y.max() - y.min())
    x1, y1, x2, y2 = x[0], y[0], x[-1], y[-1]
    denominador = math.hypot(y2 - y1, x2 - x1)
    validacion["DISTANCIA_CODO"] = (
        np.abs((y2 - y1) * x - (x2 - x1) * y + x2 * y1 - y2 * x1)
        / denominador
    )
    k_codo = int(validacion.loc[validacion["DISTANCIA_CODO"].idxmax(), "K"])
    k_silueta = int(validacion.loc[validacion["SILUETA"].idxmax(), "K"])
    validacion["K_CODO"] = k_codo
    validacion["K_SILUETA"] = k_silueta
    validacion["ES_K_CODO"] = validacion["K"].eq(k_codo)
    validacion["ES_K_MAX_SILUETA"] = validacion["K"].eq(k_silueta)
    validacion["DECISION"] = (
        "CUARTILES: silueta selecciono K=2"
        if k_silueta == 2 else f"K-MEANS: K={k_silueta}"
    )
    validacion.to_csv(
        CARPETA_SALIDA / "validacion_numero_clusters.csv",
        index=False, encoding="utf-8-sig",
    )

    fig, ejes = plt.subplots(1, 2, figsize=(13, 5))
    ejes[0].plot(validacion["K"], validacion["INERCIA"], "o-", linewidth=2)
    ejes[0].axvline(k_codo, linestyle="--", color="#E67E22", label=f"Codo: K={k_codo}")
    ejes[0].set(title="Metodo del codo", xlabel="Numero de grupos (K)", ylabel="Inercia (WCSS)")
    ejes[1].plot(validacion["K"], validacion["SILUETA"], "s-", color="#27AE60", linewidth=2)
    ejes[1].axvline(k_silueta, linestyle="--", color="#8E44AD", label=f"Maximo: K={k_silueta}")
    ejes[1].set(title="Coeficiente de silueta", xlabel="Numero de grupos (K)", ylabel="Silueta promedio")
    for ax in ejes:
        ax.set_xticks(validacion["K"])
        ax.grid(alpha=0.3)
        ax.legend()
    fig.suptitle(
        f"Seleccion de K sin {NOMBRES_DISPLAY[DEPARTAMENTO_EXCLUIDO_CLUSTERING]} (n={len(X_scaled)})",
        fontweight="bold",
    )
    plt.tight_layout(rect=[0, 0, 1, 0.94])
    plt.savefig(CARPETA_FIGURAS / "fig_anx1_validacion_k.png", dpi=300, bbox_inches="tight")
    plt.show()
    return validacion, k_silueta, k_codo


def mapa_ordinal_clusters(df: pd.DataFrame, columna: str) -> Dict[int, str]:
    medias = df.groupby(columna)["IC_PRES"].mean().sort_values()
    n = len(medias)
    return {
        cluster: f"G{i + 1} de {n} ({'menor IC' if i == 0 else 'mayor IC' if i == n - 1 else 'IC intermedio'})"
        for i, cluster in enumerate(medias.index)
    }


def prioridad_ordinal(etiqueta: str, n: int) -> str:
    numero = int(re.search(r"G(\d+)", etiqueta).group(1))
    if numero == 1:
        return f"Prioridad 1 de {n} (mayor necesidad relativa)"
    if numero == n:
        return f"Prioridad {n} de {n} (menor necesidad relativa)"
    return f"Prioridad {numero} de {n}"


def clasificar_cuartiles_modelo(modelo: pd.DataFrame) -> pd.DataFrame:
    salida = modelo.copy()
    q1, q2, q3 = salida["IC_PRES"].quantile([0.25, 0.50, 0.75])
    salida["CLASIFICACION_FINAL"] = pd.cut(
        salida["IC_PRES"],
        bins=[-np.inf, q1, q2, q3, np.inf],
        labels=["Q1 - cobertura muy baja", "Q2 - cobertura baja", "Q3 - cobertura media", "Q4 - cobertura alta"],
        include_lowest=True,
    ).astype("string")
    salida["PRIORIDAD_FINAL"] = salida["CLASIFICACION_FINAL"].map({
        "Q1 - cobertura muy baja": "Prioridad 1 de 4 (mayor necesidad relativa)",
        "Q2 - cobertura baja": "Prioridad 2 de 4",
        "Q3 - cobertura media": "Prioridad 3 de 4",
        "Q4 - cobertura alta": "Prioridad 4 de 4 (menor necesidad relativa)",
    })
    salida["UMBRAL_Q1"] = q1
    salida["UMBRAL_Q2"] = q2
    salida["UMBRAL_Q3"] = q3
    return salida


def ejecutar_clustering(
    df: pd.DataFrame,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    variables = ["C_TOTAL", "IC_PRES"]
    salida = df.copy()
    mascara = salida["DPTO_NORM"] != DEPARTAMENTO_EXCLUIDO_CLUSTERING
    salida["INCLUIDO_CLUSTERING"] = mascara
    modelo = salida.loc[mascara].copy()
    X = modelo[variables].replace([np.inf, -np.inf], np.nan)
    if X.isna().any().any():
        raise ValueError("Hay valores faltantes o infinitos en el clustering.")

    X_scaled = StandardScaler().fit_transform(X)
    validacion, k_silueta, k_codo = validar_numero_clusters(X_scaled)

    columnas_nulas = {
        "CLUSTER_KM": "Int64", "CLUSTER_HAC": "Int64",
        "CLUSTER_LABEL": "string", "CLUSTER_HAC_LABEL": "string",
        "COINCIDE_KM_HAC": "boolean", "CLASIFICACION_FINAL": "string",
        "PRIORIDAD_FINAL": "string", "METODO_SEGMENTACION": "string",
    }
    for columna, dtype in columnas_nulas.items():
        salida[columna] = pd.Series(pd.NA, index=salida.index, dtype=dtype)

    if k_silueta == 2:
        modelo = clasificar_cuartiles_modelo(modelo)
        modelo["METODO_SEGMENTACION"] = "CUARTILES (silueta K=2)"
        for columna in ["CLASIFICACION_FINAL", "PRIORIDAD_FINAL", "METODO_SEGMENTACION"]:
            salida.loc[modelo.index, columna] = modelo[columna].to_numpy()
        concordancia = modelo[["DPTO_NORM", "CLASIFICACION_FINAL", "PRIORIDAD_FINAL"]].copy()
        concordancia["K_SILUETA"] = k_silueta
        concordancia["K_CODO"] = k_codo
        concordancia["SILUETA_SELECCIONADA"] = float(validacion["SILUETA"].max())
        concordancia["CONCORDANCIA_GLOBAL"] = np.nan
        concordancia["METODO_FINAL"] = "CUARTILES"
        print("La silueta selecciono K=2; se prescinde de K-means y se usan cuartiles.")
    else:
        kmeans = KMeans(n_clusters=k_silueta, random_state=RANDOM_STATE, n_init=20)
        modelo["CLUSTER_KM"] = kmeans.fit_predict(X_scaled)
        mapa_km = mapa_ordinal_clusters(modelo, "CLUSTER_KM")
        modelo["CLUSTER_LABEL"] = modelo["CLUSTER_KM"].map(mapa_km)
        modelo["CLASIFICACION_FINAL"] = modelo["CLUSTER_LABEL"]
        modelo["PRIORIDAD_FINAL"] = modelo["CLUSTER_LABEL"].apply(
            lambda e: prioridad_ordinal(e, k_silueta)
        )
        modelo["METODO_SEGMENTACION"] = f"K-MEANS K={k_silueta}"

        Z = linkage(X_scaled, method="ward", metric="euclidean")
        modelo["CLUSTER_HAC"] = fcluster(Z, t=k_silueta, criterion="maxclust")
        mapa_hac = mapa_ordinal_clusters(modelo, "CLUSTER_HAC")
        modelo["CLUSTER_HAC_LABEL"] = modelo["CLUSTER_HAC"].map(mapa_hac)
        modelo["COINCIDE_KM_HAC"] = modelo["CLUSTER_LABEL"] == modelo["CLUSTER_HAC_LABEL"]
        concordancia_global = float(modelo["COINCIDE_KM_HAC"].mean())

        for columna in columnas_nulas:
            if columna in modelo.columns:
                salida.loc[modelo.index, columna] = modelo[columna].to_numpy()

        concordancia = modelo[[
            "DPTO_NORM", "CLUSTER_KM", "CLUSTER_LABEL", "CLUSTER_HAC",
            "CLUSTER_HAC_LABEL", "COINCIDE_KM_HAC", "CLASIFICACION_FINAL",
            "PRIORIDAD_FINAL",
        ]].copy()
        concordancia["K_SILUETA"] = k_silueta
        concordancia["K_CODO"] = k_codo
        concordancia["SILUETA_SELECCIONADA"] = float(validacion.loc[validacion["K"] == k_silueta, "SILUETA"].iloc[0])
        concordancia["CONCORDANCIA_GLOBAL"] = concordancia_global
        concordancia["METODO_FINAL"] = f"K-MEANS K={k_silueta}"

        fig, ax = plt.subplots(figsize=(12, 6))
        dendrogram(Z, labels=[NOMBRES_DISPLAY[d] for d in modelo["DPTO_NORM"]], leaf_rotation=90, ax=ax)
        ax.set_title(f"Dendrograma HAC Ward, K={k_silueta} (sin Asuncion)")
        ax.set_ylabel("Distancia Ward")
        plt.tight_layout()
        plt.savefig(CARPETA_FIGURAS / "dendrograma_hac_ward.png", dpi=300, bbox_inches="tight")
        plt.show()
        silueta_k2 = float(validacion.loc[validacion["K"] == 2, "SILUETA"].iloc[0])
        silueta_elegida = float(validacion.loc[validacion["K"] == k_silueta, "SILUETA"].iloc[0])
        print(
            f"Verificacion de la regla de contingencia K=2: silueta en K=2 = "
            f"{silueta_k2:.3f} frente a silueta maxima en K={k_silueta} = "
            f"{silueta_elegida:.3f}. La silueta NO favorecio K=2; se conserva "
            f"K-means/HAC-Ward y no se activa la clasificacion por cuartiles."
        )
        print(f"K seleccionado por silueta: {k_silueta}; K del codo: {k_codo}.")
        print(f"Concordancia ordinal K-means/HAC: {concordancia_global * 100:.1f}%.")

    concordancia["N_TERRITORIOS_MODELO"] = len(modelo)
    concordancia["EXCLUSION_METODOLOGICA"] = "Asuncion: outlier"
    print(f"Universo de segmentacion: {len(modelo)} territorios; Asuncion excluida.")
    return salida, concordancia, validacion


print("Funciones de segmentacion dinamica definidas.")


## 11. Proyecciones oficiales del INE y análisis de brechas

La población 15–29 de 2022–2032 se toma de **Estimaciones y Proyecciones
Departamentales. Revisión 2025** del INE. Para cada territorio y año se suman
los grupos oficiales 15–19, 20–24 y 25–29. No se usa regresión lineal,
crecimiento geométrico ni extrapolación propia.

La publicación nacional de “100 años de estimaciones y proyecciones” alcanza
2050, pero no desagrega los departamentos. Para esta tesis corresponde la
serie departamental 2000–2035, que cubre completamente el horizonte 2032.

El valor agregado de la tesis no es volver a proyectar la población, sino
vincular la trayectoria oficial con la oferta presencial registrada en el año
base. Se calcula una **brecha equivalente de oferta** respecto de la mediana
del IC oficial de 2022 en los 17 territorios no atípicos:

\[
IC_{d,t}=rac{CARRERAS\ PRESENCIALES_{d,2022}}{POB\ 15{-}29^{INE}_{d,t}}	imes 10.000
\]

\[
BRECHA_{d,t}=rac{IC^{ref}_{2022}	imes POB\ 15{-}29^{INE}_{d,t}}{10.000}
- CARRERAS\ PRESENCIALES_{d,2022}
\]

Una brecha positiva representa carreras presenciales equivalentes faltantes
para alcanzar el benchmark; una negativa representa superávit relativo. No es
una recomendación automática de abrir carreras: el indicador no mide cupos,
matrícula, calidad, movilidad estudiantil ni pertinencia específica.


In [ ]:
def normalizar_grupo_edad(valor: object) -> str:
    s = quitar_tildes(str(valor).upper().strip())
    s = s.replace("–", "-").replace("—", "-")
    return re.sub(r"\s+", "", s)


def cargar_proyecciones_ine_15_29(
    ruta: str,
    anio_inicio: int = 2022,
    anio_fin: int = 2032,
) -> pd.DataFrame:
    """Extrae la población oficial 15-29 del Cuadro 2 departamental del INE."""
    hoja = "C2 TP-Dpto_sex_edad"
    raw = pd.read_excel(ruta, sheet_name=hoja, header=None)
    if raw.empty:
        raise ValueError("El archivo de proyecciones departamentales esta vacio.")

    fila_anios = None
    columnas_anio: Dict[int, int] = {}
    for i, fila in raw.iterrows():
        candidatos = {}
        for columna, valor in fila.items():
            numero = convertir_numero(valor)
            if pd.notna(numero) and float(numero).is_integer() and 2000 <= int(numero) <= 2050:
                candidatos[int(numero)] = int(columna)
        if len(candidatos) >= 20:
            fila_anios = i
            columnas_anio = candidatos
            break
    if fila_anios is None:
        raise ValueError("No se encontro la fila de anos en el Cuadro 2 oficial.")

    anios = list(range(anio_inicio, anio_fin + 1))
    faltantes_anio = sorted(set(anios) - set(columnas_anio))
    if faltantes_anio:
        raise ValueError(f"El archivo no contiene los anos requeridos: {faltantes_anio}")

    grupos_objetivo = {"15-19", "20-24", "25-29"}
    registros = []
    for i in range(fila_anios + 1, len(raw) - 1):
        etiqueta = raw.iat[i, 1] if raw.shape[1] > 1 else ""
        dpto = normalizar_departamento(etiqueta)
        siguiente = normalizar_grupo_edad(raw.iat[i + 1, 1])
        if dpto not in DEPARTAMENTOS or siguiente != "0-4":
            continue

        filas_grupo: Dict[str, int] = {}
        j = i + 1
        while j < len(raw):
            grupo = normalizar_grupo_edad(raw.iat[j, 1])
            if grupo == "HOMBRES":
                break
            if grupo in grupos_objetivo:
                filas_grupo[grupo] = j
            j += 1
        faltantes_grupo = grupos_objetivo - set(filas_grupo)
        if faltantes_grupo:
            raise ValueError(f"{dpto}: faltan grupos etarios {sorted(faltantes_grupo)}")

        for anio in anios:
            poblacion = sum(
                convertir_numero(raw.iat[fila, columnas_anio[anio]])
                for fila in filas_grupo.values()
            )
            registros.append({
                "DPTO_NORM": dpto, "ANIO": anio,
                "POB_15_29_INE": int(round(poblacion)),
                "FUENTE": "INE, Estimaciones y Proyecciones Departamentales, Revision 2025",
                "GRUPOS_SUMADOS": "15-19 + 20-24 + 25-29",
            })

    salida = pd.DataFrame(registros)
    duplicados = salida.duplicated(["DPTO_NORM", "ANIO"]).any()
    if duplicados:
        raise ValueError("Se detectaron duplicados departamento-anio en la fuente oficial.")
    esperados = len(DEPARTAMENTOS) * len(anios)
    if len(salida) != esperados:
        presentes = set(salida["DPTO_NORM"]) if not salida.empty else set()
        raise ValueError(
            f"Se esperaban {esperados} filas y se obtuvieron {len(salida)}. "
            f"Territorios faltantes: {sorted(set(DEPARTAMENTOS) - presentes)}"
        )
    if (salida["POB_15_29_INE"] <= 0).any():
        raise ValueError("La proyeccion oficial contiene poblaciones no positivas.")
    print(
        f"Proyecciones INE cargadas: {len(DEPARTAMENTOS)} territorios, "
        f"{anio_inicio}-{anio_fin}, grupo 15-29."
    )
    return salida.sort_values(["DPTO_NORM", "ANIO"]).reset_index(drop=True)


def calcular_brechas_oferta(
    proyecciones_ine: pd.DataFrame,
    oferta: pd.DataFrame,
) -> pd.DataFrame:
    """Calcula brechas relativas usando oferta 2022 constante y población INE."""
    base_2022 = proyecciones_ine.loc[
        proyecciones_ine["ANIO"] == ANIO_BASE,
        ["DPTO_NORM", "POB_15_29_INE"],
    ].merge(oferta[["DPTO_NORM", "C_PRES"]], on="DPTO_NORM", how="left")
    asegurar_columnas_no_nulas(base_2022, ["POB_15_29_INE", "C_PRES"], "Base de brechas 2022")
    base_2022["IC_OFICIAL_2022"] = base_2022["C_PRES"] / base_2022["POB_15_29_INE"] * 10000
    universo = base_2022["DPTO_NORM"] != DEPARTAMENTO_EXCLUIDO_CLUSTERING
    ic_referencia = float(base_2022.loc[universo, "IC_OFICIAL_2022"].median())

    salida = proyecciones_ine.merge(
        oferta[["DPTO_NORM", "C_PRES"]], on="DPTO_NORM", how="left"
    ).merge(
        base_2022[["DPTO_NORM", "POB_15_29_INE", "IC_OFICIAL_2022"]].rename(
            columns={"POB_15_29_INE": "POB_15_29_INE_2022"}
        ),
        on="DPTO_NORM", how="left",
    )
    salida["IC_OFERTA_CONSTANTE"] = salida["C_PRES"] / salida["POB_15_29_INE"] * 10000
    salida["IC_REFERENCIA_MEDIANA_17"] = ic_referencia
    salida["OFERTA_OBJETIVO_EQ"] = ic_referencia * salida["POB_15_29_INE"] / 10000
    salida["BRECHA_NETA_EQ"] = salida["OFERTA_OBJETIVO_EQ"] - salida["C_PRES"]
    salida["DEFICIT_CARRERAS_EQ"] = np.ceil(salida["BRECHA_NETA_EQ"].clip(lower=0)).astype(int)
    salida["SUPERAVIT_CARRERAS_EQ"] = np.floor((-salida["BRECHA_NETA_EQ"]).clip(lower=0)).astype(int)
    salida["VAR_POB_15_29_DESDE_2022_PCT"] = (
        salida["POB_15_29_INE"] / salida["POB_15_29_INE_2022"] - 1
    ) * 100
    salida["VAR_IC_DESDE_2022_PCT"] = (
        salida["IC_OFERTA_CONSTANTE"] / salida["IC_OFICIAL_2022"] - 1
    ) * 100
    salida["TIPO_BRECHA"] = np.select(
        [salida["BRECHA_NETA_EQ"] > 0, salida["BRECHA_NETA_EQ"] < 0],
        ["Deficit relativo", "Superavit relativo"],
        default="En el benchmark",
    )
    salida["OFERTA_SE_MANTIENE_CONSTANTE"] = True
    salida["ADVERTENCIA"] = (
        "Indicador equivalente de oferta; no mide cupos, matricula, calidad ni demanda efectiva"
    )
    salida.loc[salida["DPTO_NORM"] == "ASUNCION", "ADVERTENCIA"] += (
        "; interpretar con cautela por movilidad estudiantil"
    )
    return salida.sort_values(["ANIO", "BRECHA_NETA_EQ"], ascending=[True, False]).reset_index(drop=True)


def resumir_brechas(brechas: pd.DataFrame, anio: int = 2032) -> pd.DataFrame:
    resumen = brechas.loc[brechas["ANIO"] == anio].copy()
    resumen["Departamento"] = resumen["DPTO_NORM"].map(NOMBRES_DISPLAY)
    columnas = [
        "Departamento", "POB_15_29_INE", "C_PRES", "IC_OFERTA_CONSTANTE",
        "IC_REFERENCIA_MEDIANA_17", "BRECHA_NETA_EQ", "DEFICIT_CARRERAS_EQ",
        "SUPERAVIT_CARRERAS_EQ", "VAR_POB_15_29_DESDE_2022_PCT", "TIPO_BRECHA",
    ]
    resumen = resumen[columnas].sort_values("BRECHA_NETA_EQ", ascending=False).reset_index(drop=True)
    print(f"\nANALISIS DE BRECHAS DE OFERTA EQUIVALENTE - {anio}")
    print(resumen.round(2).to_string(index=False))
    return resumen


print("Carga de proyecciones oficiales y analisis de brechas definidos.")


## 12. Índice de Prioridad Territorial Educativa (IPTE)

El IPTE se conserva como índice sintético de priorización administrativa. Sus
tres componentes tienen igual ponderación:

1. brecha de cobertura relativa (menor IC implica mayor prioridad);
2. brecha de pertinencia territorial (`1 - SCORE_PERTINENCIA`); y
3. presión demográfica oficial, medida mediante el crecimiento relativo de la
   población 15–29 entre 2022 y 2032 según el INE.

La normalización Min-Max utiliza como referencia los 17 territorios sin
Asunción para evitar que el outlier comprima las diferencias. Asunción se
conserva en el resultado, pero sus componentes se limitan al rango 0–1 y se
acompañan de una advertencia por movilidad estudiantil. El IPTE no utiliza
regresión, R², RMSE ni crecimiento geométrico.


In [ ]:
def normalizar_con_referencia(
    serie: pd.Series,
    mascara_referencia: pd.Series,
) -> pd.Series:
    """Normaliza 0-1 usando solo el universo de referencia y limita extremos."""
    valores = pd.to_numeric(serie, errors="coerce")
    referencia = valores.loc[mascara_referencia]
    minimo, maximo = referencia.min(), referencia.max()
    if pd.isna(minimo) or pd.isna(maximo) or maximo == minimo:
        return pd.Series(0.0, index=serie.index)
    return ((valores - minimo) / (maximo - minimo)).clip(0, 1)


def calcular_ipte_oficial(
    df_general: pd.DataFrame,
    proyecciones_ine: pd.DataFrame,
) -> pd.DataFrame:
    """Calcula el IPTE con crecimiento demografico oficial INE 2022-2032."""
    requeridas_general = ["DPTO_NORM", "IC_PRES", "SCORE_PERTINENCIA"]
    faltantes = [c for c in requeridas_general if c not in df_general.columns]
    if faltantes:
        raise ValueError(f"Faltan columnas para calcular el IPTE: {faltantes}")

    poblacion = (
        proyecciones_ine.loc[
            proyecciones_ine["ANIO"].isin([2022, 2032]),
            ["DPTO_NORM", "ANIO", "POB_15_29_INE"],
        ]
        .pivot(index="DPTO_NORM", columns="ANIO", values="POB_15_29_INE")
        .rename(columns={2022: "POB_15_29_INE_2022", 2032: "POB_15_29_INE_2032"})
        .reset_index()
    )
    salida = df_general.merge(poblacion, on="DPTO_NORM", how="left")
    asegurar_columnas_no_nulas(
        salida,
        ["IC_PRES", "SCORE_PERTINENCIA", "POB_15_29_INE_2022", "POB_15_29_INE_2032"],
        "IPTE con proyecciones oficiales",
    )

    referencia = salida["DPTO_NORM"] != DEPARTAMENTO_EXCLUIDO_CLUSTERING

    # Componente 1: menor cobertura implica mayor prioridad.
    cobertura_normalizada = normalizar_con_referencia(salida["IC_PRES"], referencia)
    salida["BRECHA_COBERTURA_N"] = 1 - cobertura_normalizada

    # Componente 2: menor pertinencia implica mayor prioridad.
    salida["BRECHA_PERTINENCIA"] = 1 - salida["SCORE_PERTINENCIA"].clip(0, 1)
    salida["BRECHA_PERTINENCIA_N"] = normalizar_con_referencia(
        salida["BRECHA_PERTINENCIA"], referencia
    )

    # Componente 3: crecimiento relativo oficial 2022-2032.
    salida["CRECIMIENTO_INE_2022_2032"] = (
        salida["POB_15_29_INE_2032"] / salida["POB_15_29_INE_2022"] - 1
    )
    salida["PRESION_DEMOGRAFICA_INE_N"] = normalizar_con_referencia(
        salida["CRECIMIENTO_INE_2022_2032"], referencia
    )

    salida["IPTE"] = (
        salida["BRECHA_COBERTURA_N"]
        + salida["BRECHA_PERTINENCIA_N"]
        + salida["PRESION_DEMOGRAFICA_INE_N"]
    ) / 3
    salida["RANK_IPTE"] = salida["IPTE"].rank(
        method="min", ascending=False
    ).astype(int)
    salida["METODO_IPTE"] = (
        "Promedio simple: brecha de cobertura + brecha de pertinencia + "
        "presion demografica oficial INE"
    )
    salida["ADVERTENCIA_IPTE"] = ""
    salida.loc[salida["DPTO_NORM"] == "ASUNCION", "ADVERTENCIA_IPTE"] = (
        "Interpretar con cautela: outlier de cobertura y alta movilidad estudiantil"
    )

    columnas_control = [
        "BRECHA_COBERTURA_N", "BRECHA_PERTINENCIA_N",
        "PRESION_DEMOGRAFICA_INE_N", "IPTE",
    ]
    if not salida[columnas_control].apply(lambda s: s.between(0, 1)).all().all():
        raise ValueError("Los componentes normalizados del IPTE deben estar entre 0 y 1.")

    salida = salida.sort_values("IPTE", ascending=False).reset_index(drop=True)
    print("\nRANKING IPTE - CRECIMIENTO OFICIAL INE 2022-2032")
    print(
        salida[[
            "RANK_IPTE", "DPTO_NORM", "IPTE", "BRECHA_COBERTURA_N",
            "BRECHA_PERTINENCIA_N", "PRESION_DEMOGRAFICA_INE_N",
            "CRECIMIENTO_INE_2022_2032",
        ]].round(3).to_string(index=False)
    )
    return salida


print("Funcion de calculo del IPTE con proyecciones oficiales definida.")


## 13. Gráficos

Se generan figuras del IC, modalidades, pertinencia y segmentación. La figura
de segmentación se adapta al método finalmente seleccionado. El componente
prospectivo muestra la trayectoria oficial del INE y la brecha neta equivalente
de oferta al 2032. También se grafica el ranking IPTE recalculado con presión
demográfica oficial; no se incluyen tendencias ajustadas ni extrapolaciones propias.


In [ ]:
def grafico_ic(df: pd.DataFrame) -> None:
    d = df.sort_values("IC_PRES", ascending=False).copy()
    colores_cuartil = {"Q1": "#d73027", "Q2": "#fc8d59", "Q3": "#fee08b", "Q4": "#1a9850", "OUTLIER": "#9E9E9E"}
    colores = d["CUARTIL"].map(colores_cuartil)
    etiquetas = [NOMBRES_DISPLAY[x] for x in d["DPTO_NORM"]]
    mediana = d.loc[d["DPTO_NORM"] != DEPARTAMENTO_EXCLUIDO_CLUSTERING, "IC_PRES"].median()

    fig, ax = plt.subplots(figsize=(13, 8))
    barras = ax.barh(etiquetas, d["IC_PRES"], color=colores)
    ax.axvline(mediana, linestyle="--", linewidth=1.5, label=f"Mediana ({mediana:.2f})")
    for barra, valor in zip(barras, d["IC_PRES"]):
        ax.text(valor + max(d["IC_PRES"]) * 0.006, barra.get_y() + barra.get_height()/2, f"{valor:.1f}", va="center", fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel("Carreras presenciales por cada 10.000 jovenes de 15 a 29 anos")
    ax.set_title("Indice de Cobertura Relativa por departamento - Ano base 2022")
    leyenda = [mpatches.Patch(color=c, label=q) for q, c in colores_cuartil.items()]
    leyenda.append(Line2D([0], [0], linestyle="--", label=f"Mediana {mediana:.2f}"))
    ax.legend(handles=leyenda, loc="lower right")
    plt.tight_layout()
    plt.savefig(CARPETA_FIGURAS / "fig_3_2_barras_ic.png", dpi=300, bbox_inches="tight")
    plt.show()


def grafico_ic_vs_mediana(df: pd.DataFrame) -> None:
    """
    Reproduce el Anexo 3 de la tesis: IC presencial coloreado en verde
    (por encima de la mediana) o rojo (por debajo).
    """
    d = df.sort_values("IC_PRES", ascending=False).copy()
    mediana = d.loc[d["DPTO_NORM"] != DEPARTAMENTO_EXCLUIDO_CLUSTERING, "IC_PRES"].median()
    colores = [
        "#9E9E9E" if dep == DEPARTAMENTO_EXCLUIDO_CLUSTERING
        else "#2ECC71" if valor >= mediana else "#E74C3C"
        for dep, valor in zip(d["DPTO_NORM"], d["IC_PRES"])
    ]
    etiquetas = [NOMBRES_DISPLAY[x] for x in d["DPTO_NORM"]]

    fig, ax = plt.subplots(figsize=(14, 6))
    ax.bar(etiquetas, d["IC_PRES"], color=colores)
    ax.axhline(mediana, color="#2C3E50", linestyle="--", linewidth=1.5)
    ax.set_xticklabels(etiquetas, rotation=45, ha="right", fontsize=9)
    ax.set_title(
        "Indice de cobertura presencial por departamento\n"
        "Carreras por cada 10.000 jovenes de 15 a 29 anos - Fuente: CONES / INE Censo 2022",
        fontweight="bold",
    )
    ax.set_ylabel("Indice de cobertura (x 10.000)")
    p1 = mpatches.Patch(color="#2ECC71", label="Por encima de la mediana")
    p2 = mpatches.Patch(color="#E74C3C", label="Por debajo de la mediana")
    p3 = Line2D([0], [0], color="#2C3E50", linestyle="--", label=f"Mediana nacional ({mediana:.1f})")
    p4 = mpatches.Patch(color="#9E9E9E", label="Asuncion: outlier")
    ax.legend(handles=[p1, p2, p3, p4])
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.tight_layout()
    plt.savefig(CARPETA_FIGURAS / "fig_anx3_cobertura_vs_mediana.png", dpi=300, bbox_inches="tight")
    plt.show()


def grafico_modalidades(df_cones: pd.DataFrame) -> None:
    conteos = df_cones["MODALIDAD"].value_counts().reindex(
        ["PRESENCIAL", "SEMIPRESENCIAL_DISTANCIA", "VIRTUAL", "OTRA"], fill_value=0
    )
    total = conteos.sum()
    etiquetas = []
    nombres = ["Presencial", "Semipresencial / a distancia", "Virtual", "Otra"]
    for nombre, valor in zip(nombres, conteos):
        etiquetas.append(f"{nombre}\n{valor/total*100:.1f}%\n({valor:,})")

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.pie(conteos.values, labels=etiquetas, startangle=90, wedgeprops={"edgecolor": "white"})
    ax.set_title(f"Distribucion de carreras activas por modalidad\nTotal: {total:,}")
    plt.tight_layout()
    plt.savefig(CARPETA_FIGURAS / "fig_3_3_modalidades.png", dpi=300, bbox_inches="tight")
    plt.show()


def grafico_segmentacion(df: pd.DataFrame, concordancia: pd.DataFrame) -> None:
    d = df.loc[df["INCLUIDO_CLUSTERING"]].dropna(subset=["CLASIFICACION_FINAL"]).copy()
    categorias = sorted(d["CLASIFICACION_FINAL"].unique())
    colores = {
        categoria: plt.cm.RdYlGn(i / max(1, len(categorias) - 1))
        for i, categoria in enumerate(categorias)
    }
    fig, ax = plt.subplots(figsize=(11, 7))
    for etiqueta, grupo in d.groupby("CLASIFICACION_FINAL"):
        ax.scatter(grupo["C_TOTAL"], grupo["IC_PRES"], s=85, color=colores[etiqueta], label=f"{etiqueta} (n={len(grupo)})")
        for _, fila in grupo.iterrows():
            ax.annotate(NOMBRES_DISPLAY[fila["DPTO_NORM"]], (fila["C_TOTAL"], fila["IC_PRES"]), xytext=(4, 2), textcoords="offset points", fontsize=7)
    metodo = str(concordancia["METODO_FINAL"].iloc[0])
    k_sil = int(concordancia["K_SILUETA"].iloc[0])
    sil = float(concordancia["SILUETA_SELECCIONADA"].iloc[0])
    conc = concordancia["CONCORDANCIA_GLOBAL"].iloc[0]
    nota = f"Silueta: K={k_sil}, {sil:.3f}"
    if pd.notna(conc):
        nota += f" | Concordancia K-means/HAC: {float(conc) * 100:.1f}%"
    ax.axhline(d["IC_PRES"].median(), linestyle="--", linewidth=1, color="#555555")
    ax.set_xlabel("Cantidad total de carreras")
    ax.set_ylabel("IC presencial por 10.000 residentes de 15-29 anos")
    ax.set_title(f"Segmentacion territorial: {metodo}\nAsuncion excluida por outlier")
    ax.legend(fontsize=8)
    ax.text(0.01, 0.01, nota, transform=ax.transAxes, fontsize=8)
    plt.tight_layout()
    plt.savefig(CARPETA_FIGURAS / "fig_3_4_segmentacion.png", dpi=300, bbox_inches="tight")
    plt.show()


def grafico_pertinencia(df: pd.DataFrame) -> None:
    d = df.sort_values("SCORE_PERTINENCIA", ascending=False)
    fig, ax = plt.subplots(figsize=(13, 6))
    ax.bar([NOMBRES_DISPLAY[x] for x in d["DPTO_NORM"]], d["SCORE_PERTINENCIA"])
    ax.axhline(0.6, linestyle="--", label="Umbral orientativo 0,6")
    ax.set_ylim(0, 1)
    ax.set_ylabel("Score de pertinencia")
    ax.set_title("Correspondencia entre oferta academica y ejes productivos")
    ax.tick_params(axis="x", rotation=55)
    ax.legend()
    plt.tight_layout()
    plt.savefig(CARPETA_FIGURAS / "fig_4_4_pertinencia.png", dpi=300, bbox_inches="tight")
    plt.show()


def grafico_evolucion_oficial_ine(
    proyecciones_ine: pd.DataFrame,
    dpto: str = "CONCEPCION",
) -> None:
    d = proyecciones_ine.loc[proyecciones_ine["DPTO_NORM"] == dpto].sort_values("ANIO")
    fig, ax = plt.subplots(figsize=(10, 5.5))
    ax.plot(d["ANIO"], d["POB_15_29_INE"], "o-", color="#2980B9", linewidth=2)
    ax.set_title(f"Demanda potencial demografica oficial 15-29 - {NOMBRES_DISPLAY[dpto]}\nINE, Revision 2025")
    ax.set_xlabel("Ano")
    ax.set_ylabel("Poblacion residente de 15 a 29 anos")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, pos: f"{x:,.0f}"))
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(CARPETA_FIGURAS / "fig_evolucion_oficial_ine_concepcion.png", dpi=300, bbox_inches="tight")
    plt.show()


def grafico_brechas_2032(brechas: pd.DataFrame) -> None:
    d = brechas.loc[brechas["ANIO"] == 2032].copy()
    d["NOMBRE"] = d["DPTO_NORM"].map(NOMBRES_DISPLAY)
    d = d.sort_values("BRECHA_NETA_EQ")
    colores = np.where(d["BRECHA_NETA_EQ"] > 0, "#D73027", "#1A9850")
    fig, ax = plt.subplots(figsize=(12, 8))
    ax.barh(d["NOMBRE"], d["BRECHA_NETA_EQ"], color=colores)
    ax.axvline(0, color="#333333", linewidth=1)
    ax.set_xlabel("Brecha neta de carreras presenciales equivalentes")
    ax.set_title("Brecha relativa de oferta al 2032\nPoblacion oficial INE y oferta 2022 constante")
    ax.grid(axis="x", alpha=0.25)
    plt.tight_layout()
    plt.savefig(CARPETA_FIGURAS / "fig_brechas_oferta_2032.png", dpi=300, bbox_inches="tight")
    plt.show()


def grafico_ipte(df_general: pd.DataFrame) -> None:
    d = df_general.sort_values("IPTE", ascending=True).copy()
    colores = plt.cm.YlOrRd(d["IPTE"].to_numpy())
    fig, ax = plt.subplots(figsize=(12, 8))
    barras = ax.barh(
        [NOMBRES_DISPLAY[dpto] for dpto in d["DPTO_NORM"]],
        d["IPTE"],
        color=colores,
    )
    for barra, valor in zip(barras, d["IPTE"]):
        ax.text(
            valor + 0.008,
            barra.get_y() + barra.get_height() / 2,
            f"{valor:.3f}",
            va="center", fontsize=8,
        )
    ax.set_xlim(0, max(1.0, float(d["IPTE"].max()) + 0.08))
    ax.set_xlabel("Indice de Prioridad Territorial Educativa (0-1)")
    ax.set_title(
        "IPTE con presion demografica oficial INE 2022-2032\n"
        "Promedio simple de tres componentes normalizados"
    )
    ax.grid(axis="x", alpha=0.25)
    plt.tight_layout()
    plt.savefig(CARPETA_FIGURAS / "fig_ranking_ipte_ine.png", dpi=300, bbox_inches="tight")
    plt.show()


print("Funciones de graficos definidas.")


## 14. Mapas

Los mapas estáticos e interactivos representan el IC, la pertinencia, la
segmentación final y la brecha neta equivalente de 2032. Asunción aparece sin
categoría en el mapa de segmentación porque fue excluida del modelado.


In [ ]:
def guardar_mapa_estatico(
    gdf,
    df: pd.DataFrame,
    columna: str,
    titulo: str,
    archivo: str,
    cmap: str = "RdYlGn",
    categorico: bool = False,
    colores_categoria: Optional[Dict[str, str]] = None,
) -> None:
    """
    Guarda un mapa coropletico estatico. Por defecto trata `columna` como
    variable continua; con categorico=True, la trata como etiqueta discreta
    (obligatorio para columnas de cluster).
    """
    if gdf is None:
        print("geopandas no disponible: se omite este mapa.")
        return
    geo = gdf.merge(df[["DPTO_NORM", columna]], on="DPTO_NORM", how="left")
    fig, ax = plt.subplots(figsize=(9, 9))

    if categorico:
        # Capa base para territorios sin clasificacion (p. ej., Asuncion).
        geo.plot(ax=ax, color="#E0E0E0", linewidth=0.6, edgecolor="white")
        categorias = list(geo[columna].dropna().unique())
        if colores_categoria is None:
            paleta_defecto = ["#d73027", "#fdae61", "#1a9850", "#4575b4", "#984ea3"]
            colores_categoria = {cat: paleta_defecto[i % len(paleta_defecto)] for i, cat in enumerate(categorias)}
        for categoria in categorias:
            subset = geo[geo[columna] == categoria]
            subset.plot(
                ax=ax,
                color=colores_categoria.get(categoria, "#999999"),
                linewidth=0.6,
                edgecolor="white",
            )
        leyenda = [
            mpatches.Patch(
                color=colores_categoria.get(cat, "#999999"), label=cat
            )
            for cat in categorias
        ]
        if geo[columna].isna().any():
            leyenda.append(
                mpatches.Patch(
                    color="#E0E0E0", label="Excluida del clustering (outlier)"
                )
            )
        ax.legend(handles=leyenda, loc="lower left", fontsize=9, title=None)
    else:
        geo.plot(column=columna, cmap=cmap, linewidth=0.6, edgecolor="white", legend=True, ax=ax)

    for _, fila in geo.iterrows():
        punto = fila.geometry.representative_point()
        ax.annotate(NOMBRES_DISPLAY[fila["DPTO_NORM"]], (punto.x, punto.y), ha="center", fontsize=6)
    ax.set_title(titulo)
    ax.axis("off")
    plt.tight_layout()
    plt.savefig(CARPETA_FIGURAS / archivo, dpi=300, bbox_inches="tight")
    plt.show()


def guardar_mapa_folium(gdf, df: pd.DataFrame, columna: str, leyenda: str, archivo: str, fill_color: str = "YlOrRd") -> None:
    if gdf is None or folium is None:
        print("geopandas/folium no disponibles: se omite este mapa.")
        return
    geojson = json.loads(gdf.to_json())
    mapa = folium.Map(location=[-23.4, -58.4], zoom_start=6, tiles="CartoDB positron")
    folium.Choropleth(
        geo_data=geojson,
        data=df,
        columns=["DPTO_NORM", columna],
        key_on="feature.properties.DPTO_NORM",
        fill_color=fill_color,
        fill_opacity=0.75,
        line_opacity=0.3,
        legend_name=leyenda,
    ).add_to(mapa)

    tooltip_df = df.set_index("DPTO_NORM")
    for feature in geojson["features"]:
        dpto = feature["properties"]["DPTO_NORM"]
        valor = tooltip_df.loc[dpto, columna]
        folium.GeoJson(
            feature,
            style_function=lambda _: {"fillOpacity": 0, "color": "transparent"},
            tooltip=folium.Tooltip(f"<b>{NOMBRES_DISPLAY[dpto]}</b><br>{leyenda}: {valor:,.3f}"),
        ).add_to(mapa)
    mapa.save(str(CARPETA_MAPAS / archivo))
    print(f"Mapa interactivo guardado en: {CARPETA_MAPAS / archivo}")


print("Funciones de mapas definidas; las brechas 2032 usan los mapas genericos.")


## 15. Exportación de resultados

Se exportan la población oficial 15–29 del INE, el análisis anual de brechas,
la tabla de brechas 2032, el ranking IPTE, la decisión de K, la concordancia
metodológica y los resultados descriptivos. No se generan archivos de regresión,
comparación de modelos ni sensibilidad de parámetros.


In [ ]:
def exportar_resultados(
    df_general: pd.DataFrame,
    por_area: pd.DataFrame,
    proyecciones_ine: pd.DataFrame,
    brechas: pd.DataFrame,
    concordancia: pd.DataFrame,
    validacion_k: pd.DataFrame,
    moran: pd.DataFrame,
    pertinencia: pd.DataFrame,
    diversidad: pd.DataFrame,
    ranking_divergencia: pd.DataFrame,
) -> None:
    columnas_ipte = [
        "RANK_IPTE", "DPTO_NORM", "IPTE", "BRECHA_COBERTURA_N",
        "BRECHA_PERTINENCIA_N", "PRESION_DEMOGRAFICA_INE_N",
        "CRECIMIENTO_INE_2022_2032", "POB_15_29_INE_2022",
        "POB_15_29_INE_2032", "ADVERTENCIA_IPTE",
    ]
    faltantes_ipte = [c for c in columnas_ipte if c not in df_general.columns]
    if faltantes_ipte:
        raise ValueError(f"Faltan columnas del IPTE para exportar: {faltantes_ipte}")
    ranking_ipte = df_general[columnas_ipte].sort_values("RANK_IPTE").copy()

    archivos = {
        "resultado_general_departamentos.csv": df_general,
        "resultado_por_area_departamentos.csv": por_area,
        "proyecciones_ine_15_29_oficial_2022_2032.csv": proyecciones_ine,
        "analisis_brechas_oferta_2022_2032.csv": brechas,
        "concordancia_segmentacion.csv": concordancia,
        "validacion_numero_clusters.csv": validacion_k,
        "indice_moran.csv": moran,
        "pertinencia_territorial.csv": pertinencia,
        "diversidad_departamental.csv": diversidad,
        "tabla_3_2_ranking_absoluto_vs_ic.csv": ranking_divergencia,
        "ranking_ipte_proyeccion_oficial_ine.csv": ranking_ipte,
    }
    for nombre, tabla in archivos.items():
        tabla.to_csv(CARPETA_SALIDA / nombre, index=False, encoding="utf-8-sig")

    tabla_2032 = brechas.loc[brechas["ANIO"] == 2032].copy()
    tabla_2032.to_csv(
        CARPETA_SALIDA / "tabla_4_8_brechas_oferta_2032.csv",
        index=False, encoding="utf-8-sig",
    )
    tabla_2032.loc[tabla_2032["DPTO_NORM"] != "ASUNCION"].to_csv(
        CARPETA_SALIDA / "tabla_4_8_brechas_oferta_2032_sin_asuncion.csv",
        index=False, encoding="utf-8-sig",
    )
    print(f"Todos los CSV fueron guardados en: {CARPETA_SALIDA.resolve()}")


print("Funcion de exportacion definida.")


## 16. Ejecución del pipeline completo

La secuencia carga las cuatro fuentes, prepara la oferta, calcula indicadores,
selecciona el método de segmentación y finalmente aplica las proyecciones
oficiales del INE al análisis de brechas.


### Paso 1/9 — Cargar las cuatro fuentes de datos

Se cargan los cuatro insumos originales del estudio: el Registro Nacional de
Carreras del CONES (\texttt{carreras.txt}), la población total por
departamento del Cuadro 1 del Censo 2022 del INE (\texttt{Cuadro\_1.xlsx}),
el GeoJSON departamental para los mapas, y las Estimaciones y Proyecciones
Departamentales del INE (Revisión 2025), de donde se extraerá más adelante el
grupo de 15 a 29 años para el período 2022–2032. Ningún dato se modifica en
este paso; solo se leen los archivos y se estandarizan los nombres de columna
y de departamento para poder cruzarlos entre sí.


In [ ]:
crear_carpetas()
print("=" * 78)
print("PIPELINE - OFERTA DE EDUCACION SUPERIOR Y BRECHAS TERRITORIALES")
print("=" * 78)

print("\n[1/9] Cargando fuentes...")
cones_raw = cargar_cones(RUTA_CONES)
pob_2022 = cargar_ine_cuadro1(RUTA_INE_CUADRO_1)
proyecciones_ine = cargar_proyecciones_ine_15_29(RUTA_INE_PROYECCIONES)
gdf = cargar_geojson(RUTA_GEOJSON)


### Paso 2/9 — Preparar y clasificar la oferta CONES

Se depuran los registros del CONES (se descartan carreras inactivas o con
campos de departamento/modalidad vacíos, sin eliminar duplicados legítimos de
distintas instituciones) y se clasifican por modalidad de enseñanza, área
disciplinaria (Frascati) y área estratégica. El resultado son los conteos de
carreras por departamento que alimentan el Índice de Cobertura Relativa (IC)
en el paso siguiente.


In [ ]:
print("\n[2/9] Preparando oferta CONES...")
cones = preparar_cones(cones_raw)
auditoria_clasificacion = exportar_auditoria_clasificacion(cones)
print("\nDetalle de clasificacion - departamento de \u00d1eembucu:")
print(
    auditoria_clasificacion.loc[
        auditoria_clasificacion["DPTO_NORM"] == "NEEMBUCU",
        ["NOMBRE_CARRERA", "MODALIDAD", "AREA_ESTRATEGICA", "EJES_ESTRATEGICOS"],
    ].to_string(index=False)
)
oferta, oferta_area = construir_oferta(cones)


### Paso 3/9 — Construir demanda potencial e indicadores

Se define la demanda potencial exclusivamente como la población residente de
15 a 29 años (Censo 2022) y se construye el Índice de Cobertura Relativa (IC):
carreras presenciales por cada 10.000 jóvenes de cada departamento. También se
calculan el score de pertinencia territorial (correspondencia entre la oferta
académica y los ejes productivos) y el índice de diversidad disciplinar. Estos
indicadores son la base de todos los análisis posteriores.


In [ ]:
print("\n[3/9] Construyendo demanda e indicadores...")
demanda = construir_demanda(pob_2022)
general = oferta.merge(demanda, on="DPTO_NORM", how="left")
general = construir_indices(general)
ranking_divergencia = calcular_tabla_ranking_divergencia(general)

diversidad = calcular_diversidad_hhi(cones)
pertinencia = calcular_pertinencia(cones)
general = general.merge(diversidad, on="DPTO_NORM", how="left")
general = general.merge(pertinencia, on="DPTO_NORM", how="left")
validar_departamentos(general, "Resultado general")


### Paso 4/9 — Análisis exploratorio (EDA) e Índice de Moran

Se calculan estadísticos descriptivos del IC (media, mediana, cuartiles) y se
identifica a Asunción como valor atípico mediante su puntaje Z. Adicionalmente
se calcula el Índice de Moran sobre el IC de los 18 territorios, usando una
matriz de contigüidad tipo Queen, para evaluar si existe autocorrelación
espacial (es decir, si los departamentos con cobertura similar tienden a
agruparse geográficamente).


In [ ]:
print("\n[4/9] Ejecutando EDA e Indice de Moran...")
general = ejecutar_eda(general)
moran = calcular_moran(gdf, general)


### Paso 5/9 — Seleccionar la segmentación territorial sin Asunción

Asunción se excluye antes de estandarizar las variables, por su condición de
valor atípico. Sobre los 17 departamentos restantes se evalúa K = 2, ..., 8
mediante el método del codo y el coeficiente de silueta mediante
\texttt{validar\_numero\_clusters}.

Se aplica una regla de contingencia establecida de antemano: si la silueta
favorece K = 2 (una partición binaria de escaso valor analítico dado el
número reducido de territorios comparables), se prescinde de K-means/HAC-Ward
y la clasificación se apoya en los cuartiles del IC ya calculados en el
Paso 3/9. Si la silueta favorece K ≥ 3, se conserva K-means, contrastado con
clustering jerárquico aglomerativo (HAC, enlace de Ward) para verificar la
estabilidad de la segmentación. El bloque de código imprime explícitamente el
resultado de esa verificación antes de continuar.


In [ ]:
print("\n[5/9] Seleccionando segmentacion territorial sin Asuncion...")
general, concordancia, validacion_k = ejecutar_clustering(general)
tabla_4_3 = generar_tabla_4_3(general)


### Paso 6/9 — Calcular brechas con la proyección oficial del INE

La oferta presencial observada en 2022 se mantiene constante entre 2022 y
2032 para aislar el efecto de la trayectoria demográfica oficial del INE
sobre el IC. Es un escenario analítico deliberadamente conservador para
construir una referencia común y reproducible; no es una predicción de la
oferta académica futura ni anticipa decisiones del CONES o de las
instituciones educativas.


In [ ]:
print("\n[6/9] Calculando brechas con proyecciones oficiales del INE...")
brechas = calcular_brechas_oferta(proyecciones_ine, oferta)
brechas_2032 = resumir_brechas(brechas, anio=2032)


### Paso 7/9 — Revisar los territorios con mayor déficit relativo

Se ordenan los departamentos según la brecha equivalente de oferta calculada
en el paso anterior, de mayor a menor déficit hacia 2032. Este ranking orienta
dónde conviene profundizar el diagnóstico institucional; no sustituye estudios
de matrícula, capacidad, movilidad estudiantil, calidad académica ni
pertinencia de carreras concretas.


In [ ]:
print("\n[7/9] Territorios con mayor deficit equivalente en 2032...")
print(
    brechas_2032.loc[brechas_2032["DEFICIT_CARRERAS_EQ"] > 0].head(10).to_string(index=False)
)


### Paso 8/9 — Calcular el IPTE con presión demográfica oficial del INE

El Índice de Prioridad Territorial Educativa (IPTE) combina tres componentes
normalizados con ponderación uniforme de un tercio: la brecha de cobertura
(IC insuficiente), la brecha de pertinencia (correspondencia insuficiente con
los ejes productivos) y la presión demográfica oficial 2022–2032 del INE.
Asunción queda excluida de la normalización, el cálculo y el ranking. El
propósito del índice es ordenar prioridades relativas entre los 17
departamentos, no predecir matrícula ni sustituir el análisis sustantivo de
cada territorio.


In [ ]:
print("\n[8/9] Calculando IPTE con proyecciones oficiales del INE...")
general = calcular_ipte_oficial(general, proyecciones_ine)
ranking_ipte = general[[
    "RANK_IPTE", "DPTO_NORM", "IPTE", "BRECHA_COBERTURA_N",
    "BRECHA_PERTINENCIA_N", "PRESION_DEMOGRAFICA_INE_N",
    "CRECIMIENTO_INE_2022_2032",
]].sort_values("RANK_IPTE")


### Paso 9/9 — Generar figuras, mapas y archivos de resultados

Se producen todas las figuras descriptivas del estudio (distribución del IC,
modalidades, segmentación con K=5, pertinencia, trayectoria demográfica
oficial, brechas 2032 e IPTE), los mapas estáticos e interactivos, y se
exportan todos los CSV reproducibles a la carpeta \texttt{resultados\_tesis/}
mediante \texttt{exportar\_resultados}. Este es el último paso obligatorio
del pipeline; el paso opcional siguiente solo empaqueta estos resultados para
descargarlos de una sola vez.


In [ ]:
print("\n[9/9] Generando figuras, mapas y CSV...")
grafico_ic(general)
grafico_ic_vs_mediana(general)
grafico_modalidades(cones)
grafico_segmentacion(general, concordancia)
grafico_pertinencia(general)
grafico_evolucion_oficial_ine(proyecciones_ine, dpto="CONCEPCION")
grafico_brechas_2032(brechas)
grafico_ipte(general)

guardar_mapa_estatico(gdf, general, "IC_PRES", "Indice de Cobertura Presencial", "fig_4_1_mapa_cobertura.png", cmap="RdYlGn")
guardar_mapa_estatico(gdf, general, "SCORE_PERTINENCIA", "Pertinencia territorial", "fig_4_3_mapa_pertinencia.png", cmap="RdYlGn")
guardar_mapa_estatico(
    gdf, general, "CLASIFICACION_FINAL", "Segmentacion territorial (Asuncion excluida)",
    "fig_4_2_mapa_segmentacion.png", categorico=True,
)
brechas_mapa_2032 = brechas.loc[brechas["ANIO"] == 2032].copy()
guardar_mapa_estatico(
    gdf, brechas_mapa_2032, "BRECHA_NETA_EQ",
    "Brecha neta de oferta equivalente al 2032", "fig_mapa_brecha_2032.png", cmap="RdYlGn_r",
)
guardar_mapa_estatico(
    gdf, general, "IPTE", "Indice de Prioridad Territorial Educativa",
    "fig_mapa_ipte_ine.png", cmap="YlOrRd",
)

guardar_mapa_folium(gdf, general, "IC_PRES", "Cobertura presencial", "mapa_cobertura_presencial.html", fill_color="RdYlGn")
guardar_mapa_folium(gdf, brechas_mapa_2032, "BRECHA_NETA_EQ", "Brecha neta equivalente 2032", "mapa_brecha_2032.html", fill_color="YlOrRd")
guardar_mapa_folium(gdf, general, "IPTE", "IPTE con crecimiento oficial INE", "mapa_ipte_ine.html", fill_color="YlOrRd")

exportar_resultados(
    general, oferta_area, proyecciones_ine, brechas, concordancia,
    validacion_k, moran, pertinencia, diversidad, ranking_divergencia,
)
print("\nPipeline finalizado correctamente.")
print(f"Resultados guardados en: {CARPETA_SALIDA.resolve()}")


### (Opcional) Descargar todos los resultados como un .zip

Esta celda comprime toda la carpeta \texttt{resultados\_tesis/} (CSV, figuras
y mapas) en un único archivo \texttt{.zip} usando \texttt{shutil.make\_archive},
y luego lo descarga al equipo local mediante \texttt{colab\_files.download}. Es
una manera de obtener de una sola vez todo lo generado por el pipeline, sin
tener que descargar archivo por archivo desde el panel de archivos de Colab.
Este paso es opcional y no forma parte de los 9 pasos obligatorios del
pipeline.


In [ ]:
import shutil
from google.colab import files as colab_files

nombre_zip = "resultados_tesis"
shutil.make_archive(nombre_zip, "zip", CARPETA_SALIDA)
colab_files.download(f"{nombre_zip}.zip")
